In [ ]:
import glob, os, re, json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from scipy.stats import ttest_rel, pearsonr
from scipy.io import loadmat
from IPython.display import display, Math, Latex

band_names = [
    r"$\delta$",
    r"$\theta$",
    r"$\alpha$",
    r"$\beta_{LOW}$",
    r"$\beta_{MID}$",
    r"$\beta_{HIGH}$",
    r"$\gamma$",
    r"$\beta_{ALL}$",
    r"$1-40 Hz$",
]

# Amount of non-linearity

## fMRI

In [ ]:
# loading fMRI data
listheo = []
listrue = []
fmri_res = {}

# for folder in glob.glob("../../NonLinearityResultsNew/eso245_cra_strin_*_Extended_bin8"):
for folder in glob.glob(
    "../../NonLinearityResultsNew/shaved_eso245_cra_strin_*_Extended_bin8"
):
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        statsMI = np.load(os.path.join(folder, f"subject00_8.npy"))
        pairs = statsMI.shape[0]
        regions = int(0.5 + np.sqrt(0.25 + 2 * pairs))
        theo = int(re.findall(r"_(\d+)", folder)[0])
        if theo not in listheo:
            listheo.append(theo)
            listrue.append(regions)
        print(folder, theo, regions, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            fmri_res[regions] = {k: np.array(v) for k, v in json.load(fp).items()}
listheo.append(18380)
listrue.append(18380)
tru2theo = {k: v for k, v in zip(listrue, listheo)}
theo2tru = {v: k for k, v in zip(listrue, listheo)}
with open(
    "../../NonLinearityResultsNew/shaved_100center_Extended_bin8/shaved_100center_Extended_bin8_globalStats.json"
) as fp:
    fmri_res[18380] = {k: np.array(v) for k, v in json.load(fp).items()}
# del statsReg[586]
order = np.argsort(listheo)
plt.plot(np.array(listheo)[order], np.array(listrue)[order], "o--")
plt.plot(np.array(listheo)[order], np.array(listheo)[order], ":k", lw=0.5)
plt.xlabel("Designed number of regions")
plt.ylabel("Effective number of regions")
plt.xscale("log")
plt.yscale("log")
plt.show()

In [ ]:
for i in fmri_res:
    fmri_res[i]["RNL"] = (fmri_res[i]["totalMI"] - fmri_res[i]["gaussMI"]) / fmri_res[
        i
    ]["totalMI"]
    fmri_res[i]["RNLshadow"] = (
        fmri_res[i]["totalMIshadow"] - fmri_res[i]["gaussMIshadow"]
    ) / fmri_res[i]["totalMIshadow"]

In [ ]:
fig = plt.figure(figsize=(12, 6))
x = sorted(fmri_res.keys())  # [theo2tru[k] for k in sorted(theo2tru.keys())]#
STRETCH = 1.8

trueRNL = [fmri_res[i]["RNL"] for i in x]
plt.boxplot(
    trueRNL,
    positions=np.arange(len(x)) * STRETCH - 0.3,
    boxprops={"color": "darkorange"},
    whiskerprops={"color": "darkorange"},
    flierprops={"markeredgecolor": "darkorange", "marker": "+"},
    medianprops={"lw": 1.6, "color": "darkorange"},
    capprops={"color": "darkorange"},
)
shadowRNL = [fmri_res[i]["RNLshadow"] for i in x]
plt.boxplot(
    shadowRNL,
    positions=np.arange(len(x)) * STRETCH + 0.3,
    boxprops={"color": "blue"},
    whiskerprops={"color": "blue"},
    flierprops={"markeredgecolor": "blue", "marker": "+"},
    medianprops={"lw": 1.6, "color": "blue"},
    capprops={"color": "blue"},
)

tkpos = np.arange(len(x)) * STRETCH + 0
tickNames = [
    f"{n}\n({theo2tru[n]})" for n in sorted(listheo)
]  # sorted(theo2tru.keys())]
tickNames[-1] = "Sing.\nVoxel"
plt.xticks(tkpos, tickNames, rotation=0, fontsize="small")

(p1,) = plt.plot([-0.5, len(x) * STRETCH - 1], [0, 0], "-r", lw=1, label=0, zorder=1)
p2 = plt.hlines(
    0.05, -0.5, len(x) * STRETCH - 1, "r", "dashed", lw=1, label="5%", zorder=1
)
plt.ylabel("Relative non-linearity in MI")
plt.xlabel("# of regions")
plt.legend(
    [
        Patch(facecolor="white", edgecolor="darkorange"),
        Patch(facecolor="white", edgecolor="blue"),
        p1,
        p2,
    ],
    ["Total MI vs Gauss MI", "Shadow dataset", r"0%", r"5%"],
)
plt.title(f"{len(fmri_res[10]['totalMI'])} subjects $-$ rs-fMRI")
plt.savefig("OHBM_fMRISha.pdf", bbox_inches="tight")
plt.ylim((-0.05, 0.12))
plt.show()

In [ ]:
A = np.array([k for k, a in zip(sorted(fmri_res.keys()), trueRNL) if len(a) == 242])
trueRNL = [a for a in trueRNL if len(a) == 242]
shadowRNL = [a for a in shadowRNL if len(a) == 242]

In [ ]:
sig_diff_reg = np.array([ttest_rel(T, S).pvalue for T, S in zip(trueRNL, shadowRNL)])
avTrueRNL_fMRI = np.mean(trueRNL, axis=1)
avShadRNL_fMRI = np.mean(shadowRNL, axis=1)
palette = plt.cm.magma
palette.set_bad(color="black")  # Bad values (i.e., masked, set to black 0.8
palette.set_under(color="grey")  # Bad values (i.e., masked, set to black 0.8
palette.set_over(color="yellow")  # Bad values (i.e., masked, set to black 0.8
# A = np.array(sorted(fmri_res.keys()), dtype=float)#sorted(theo2tru.keys())

fig, ax = plt.subplots(1, 2, figsize=(9, 5))
plt.sca(ax[0])
# A[sig_diff_reg>0.05]=np.nan
plt.scatter(avTrueRNL_fMRI, avShadRNL_fMRI, c=A, cmap=palette, vmax=1000, vmin=15)
cb = plt.colorbar(extend="max")
cb.ax.set_ylabel("# regions", rotation=90)
# plt.plot([max(min(avTrueRNL_fMRI),min(avShadRNL_fMRI))]*2, [min(max(avTrueRNL_fMRI),max(avShadRNL_fMRI))]*2, ":k", lw=1)
plt.xlabel("Data RNL")
plt.ylabel("Shadow RNL")

plt.sca(ax[1])
plt.scatter(A, avTrueRNL_fMRI, label="True")
plt.scatter(A, avShadRNL_fMRI, label="Shadow")
plt.scatter(A, avTrueRNL_fMRI - avShadRNL_fMRI, label="Difference", marker="+")
plt.xscale("log")
plt.xlabel("# regions")
plt.ylabel("RNL")
plt.legend(loc=0)
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
print(A[np.argmax(avTrueRNL_fMRI - avShadRNL_fMRI)])
plt.suptitle("Relationship between true and shadow $-$ rs-fMRI")
plt.show()

In [ ]:
sig_diff_reg = np.array([ttest_rel(T, S).pvalue for T, S in zip(trueRNL, shadowRNL)])
avTrueRNL_fMRI = np.mean(trueRNL, axis=1)
avShadRNL_fMRI = np.mean(shadowRNL, axis=1)
palette = plt.cm.magma
palette.set_bad(color="black")  # Bad values (i.e., masked, set to black 0.8
palette.set_under(color="grey")  # Bad values (i.e., masked, set to black 0.8
palette.set_over(color="yellow")  # Bad values (i.e., masked, set to black 0.8
# A = np.array(sorted(fmri_res.keys()), dtype=float)#sorted(theo2tru.keys())

fig, ax = plt.subplots(1, 2, figsize=(9, 5))
plt.sca(ax[0])
# A[sig_diff_reg>0.05]=np.nan
plt.plot(avTrueRNL_fMRI[1:-1], avShadRNL_fMRI[1:-1])
plt.scatter(
    avTrueRNL_fMRI[1:-1],
    avShadRNL_fMRI[1:-1],
    c=np.log10(A[1:-1]),
    cmap=palette,
    vmax=3,
    vmin=1,
)
cb = plt.colorbar(extend="max")
cb.ax.set_ylabel("# regions", rotation=90)
# plt.plot([max(min(avTrueRNL_fMRI),min(avShadRNL_fMRI))]*2, [min(max(avTrueRNL_fMRI),max(avShadRNL_fMRI))]*2, ":k", lw=1)
plt.xlabel("Data RNL")
plt.ylabel("Shadow RNL")

plt.sca(ax[1])
plt.scatter(A, avTrueRNL_fMRI, label="True")
plt.scatter(A, avShadRNL_fMRI, label="Shadow")
plt.scatter(A, avTrueRNL_fMRI - avShadRNL_fMRI, label="Difference", marker="+")
plt.xscale("log")
plt.xlabel("# regions")
plt.ylabel("RNL")
plt.legend(loc=0)
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
print(A[np.argmax(avTrueRNL_fMRI - avShadRNL_fMRI)])
plt.suptitle("Relationship between true and shadow $-$ rs-fMRI")
plt.show()
pearsonr(avTrueRNL_fMRI[4:-4], avShadRNL_fMRI[4:-4]), A[:4], A[-4:]

In [ ]:
np.mean(avShadRNL_fMRI[4:-4])

In [ ]:
palette = plt.cm.magma
palette.set_bad(color="black")  # Bad values (i.e., masked, set to black 0.8
palette.set_under(color="grey")  # Bad values (i.e., masked, set to black 0.8
palette.set_over(color="yellow")  # Bad values (i.e., masked, set to black 0.8
A = np.array(sorted(fmri_res.keys()), dtype=float)  # sorted(theo2tru.keys())

fig, ax = plt.subplots(1, 1, figsize=(9, 9))
plt.sca(ax)
# A[sig_diff_reg>0.05]=np.nan
for i in range(24):
    plt.scatter(
        trueRNL[i],
        shadowRNL[i],
        c=np.ones(242) * A[i],
        cmap=palette,
        vmax=1000,
        vmin=15,
        s=5,
    )
ax.set_aspect(1)
pts = [max(plt.xlim()[0], plt.ylim()[0]), min(plt.xlim()[1], plt.ylim()[1])]
plt.xlim(plt.xlim())
plt.ylim(plt.ylim())
plt.plot(pts, pts, ":k", lw=1)
plt.xlabel("Data RNL")
plt.ylabel("Shadow RNL")
plt.show()

palette.set_over(color="grey")  # Bad values (i.e., masked, set to black 0.8
corr_fmri = [pearsonr(trueRNL[i], shadowRNL[i]) for i in range(24)]
plt.scatter(
    A,
    [q[0] for q in corr_fmri],
    c=[q[1] for q in corr_fmri],
    cmap=palette,
    vmax=0.05,
    vmin=0,
)
plt.xscale("log")
plt.show()

## EEG

**Fixed time** corresponds to *45 s*

**Fixed samples** is *1000 samples*

In particular delta and theta fixed time bands have less than 1000 samples (405, 810).

In [ ]:
for b in range(1, 10):
    mat = loadmat(
        f"../../NonLinearityData/EEG_el_so_BLP_NEW/NEW_EEG_fixedTime_band{b}.mat"
    )["EEG"]
    display(Latex(band_names[b - 1] + f" = {mat.shape[0]}"))

### Scalp Voltage

In [ ]:
eeg_scalp_volt_Samp = {}
for folder in glob.glob(
    "../../NonLinearityResultsNew/NEW_EEG_fixedSamples_band?_bin10"
):
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        try:
            statsMI = np.load(os.path.join(folder, f"subject00_10.npy"))
        except:
            statsMI = np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"band(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            eeg_scalp_volt_Samp[band] = {
                k: np.array(v) for k, v in json.load(fp).items()
            }

eeg_scalp_volt_Time = {}
for folder in glob.glob(
    "../../NonLinearityResultsNew/trimmed_EEG_fixedTime_band?_bin*"
):
    bins = int(re.findall("bin(\d+)", folder)[0])
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        try:
            statsMI = np.load(os.path.join(folder, f"subject00_{bins}.npy"))
        except:
            statsMI = np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"band(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            eeg_scalp_volt_Time[band] = {
                k: np.array(v) for k, v in json.load(fp).items()
            }

In [ ]:
for i in eeg_scalp_volt_Samp:
    eeg_scalp_volt_Samp[i]["RNL"] = (
        eeg_scalp_volt_Samp[i]["totalMI"] - eeg_scalp_volt_Samp[i]["gaussMI"]
    ) / eeg_scalp_volt_Samp[i]["totalMI"]
    eeg_scalp_volt_Samp[i]["RNLshadow"] = (
        eeg_scalp_volt_Samp[i]["totalMIshadow"]
        - eeg_scalp_volt_Samp[i]["gaussMIshadow"]
    ) / eeg_scalp_volt_Samp[i]["totalMIshadow"]
    eeg_scalp_volt_Time[i]["RNL"] = (
        eeg_scalp_volt_Time[i]["totalMI"] - eeg_scalp_volt_Time[i]["gaussMI"]
    ) / eeg_scalp_volt_Time[i]["totalMI"]
    eeg_scalp_volt_Time[i]["RNLshadow"] = (
        eeg_scalp_volt_Time[i]["totalMIshadow"]
        - eeg_scalp_volt_Time[i]["gaussMIshadow"]
    ) / eeg_scalp_volt_Time[i]["totalMIshadow"]

In [ ]:
STRETCH = 1.5
x = sorted(eeg_scalp_volt_Samp.keys())
basepos = np.arange(len(x)) * STRETCH
fig, ax = plt.subplots(1, 2, figsize=(9, 5), sharey=True)

for i, extra in enumerate(["", "shadow"]):
    plt.sca(ax[i])
    pos = basepos - 0.33
    y = [eeg_scalp_volt_Time[i]["RNL" + extra] for i in x]
    plt.boxplot(
        y,
        positions=pos,
        widths=0.6,
        boxprops={"color": "blue"},
        whiskerprops={"color": "blue"},
        flierprops={"markeredgecolor": "blue", "marker": "+"},
        medianprops={"lw": 1.6, "color": "blue"},
        capprops={"color": "blue"},
    )

    pos = basepos + 0.33
    y = [eeg_scalp_volt_Samp[i]["RNL" + extra] for i in x]
    plt.boxplot(
        y,
        positions=pos,
        widths=0.6,
        boxprops={"color": "darkorange"},
        whiskerprops={"color": "darkorange"},
        flierprops={"markeredgecolor": "darkorange", "marker": "+"},
        medianprops={"lw": 1.6, "color": "darkorange"},
        capprops={"color": "darkorange"},
    )

    p1 = plt.hlines(
        0.0,
        basepos[0] - 0.6,
        basepos[-1] + 0.6,
        "r",
        "solid",
        lw=1,
        label="%",
        zorder=1,
    )
    p2 = plt.hlines(
        0.1,
        basepos[0] - 0.6,
        basepos[-1] + 0.6,
        "r",
        "dotted",
        lw=1,
        label="10%",
        zorder=1,
    )

    plt.ylabel(f"relative difference in {extra} MI")
    plt.xlabel("Band")
    plt.xticks(ticks=basepos, labels=band_names)
    plt.title("Shadow" if extra else "True")
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
plt.legend(
    [
        Patch(facecolor="white", edgecolor="blue"),
        Patch(facecolor="white", edgecolor="darkorange"),
        p1,
        p2,
    ],
    ["Fixed time", "Fixed #samples", r"0", r"10%"],
    loc=0,
)
plt.suptitle("50 subjects $-$ Scalp Voltage")
plt.show()

In [ ]:
sigi = np.array(
    [
        ttest_rel(a, b, alternative="greater").pvalue
        for a, b in zip(
            [eeg_scalp_volt_Samp[i]["RNL"] for i in x],
            [eeg_scalp_volt_Samp[i]["RNLshadow"] for i in x],
        )
    ]
)
print(sigi)
as_sigi = np.argsort(sigi)
HB = sigi[as_sigi] < 0.05 / len(sigi)  # /(len(sigi)-np.arange(len(sigi)))
aster = as_sigi[HB]
0.05 / (len(sigi) - np.arange(len(sigi))), aster

In [ ]:
STRETCH = 1.5
x = sorted(eeg_scalp_volt_Samp.keys())
basepos = np.arange(5) * STRETCH
fig, ax = plt.subplots(1, 1, figsize=(9, 7), sharey=True)

plt.sca(ax)
pos = basepos - 0.33
for i, (extra, color) in enumerate([("", "darkorange"), ("shadow", "blue")]):
    y = [eeg_scalp_volt_Samp[i]["RNL" + extra] for i in [1, 2, 3, 8, 7]]
    plt.boxplot(
        y,
        positions=pos,
        widths=0.6,
        boxprops={"color": color},
        whiskerprops={"color": color},
        flierprops={"markeredgecolor": color, "marker": "+"},
        medianprops={"lw": 1.6, "color": color},
        capprops={"color": color},
    )
    pos = basepos + 0.33

p1 = plt.hlines(
    0.0, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "solid", lw=1, label="%", zorder=1
)
p2 = plt.hlines(
    0.01, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "dashed", lw=1, label="1%", zorder=1
)
p3 = plt.hlines(
    0.1, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "dotted", lw=1, label="10%", zorder=1
)
# ast = plt.scatter(basepos[aster], np.full_like(aster, 0.15, dtype=float), marker="*", color=(0,0,0))
ast = plt.scatter(basepos[:3], [0.15, 0.15, 0.15], marker="*", color=(0, 0, 0))
plt.ylabel(f"Relative Difference in MI")
plt.xlabel("Band")
band_names2 = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
plt.xticks(ticks=basepos, labels=band_names2)
plt.legend(
    [
        Patch(facecolor="white", edgecolor="darkorange"),
        Patch(facecolor="white", edgecolor="blue"),
        p1,
        p2,
        p3,
        ast,
    ],
    ["True", "Shadow", r"0%", r"1%", r"10%", "Significative\ndifference\n(Bonferroni)"],
    loc=0,
)
# plt.title("50 subjects $-$ Scalp Voltage")
plt.show()

In [ ]:
# tab20
avTrueRNL_EEG_scalp_volt_samp = np.mean(
    [eeg_scalp_volt_Samp[i]["RNL"] for i in x], axis=1
)
avShadRNL_EEG_scalp_volt_samp = np.mean(
    [eeg_scalp_volt_Samp[i]["RNLshadow"] for i in x], axis=1
)
avTrueRNL_EEG_scalp_volt_time = np.mean(
    [eeg_scalp_volt_Time[i]["RNL"] for i in x], axis=1
)
avShadRNL_EEG_scalp_volt_time = np.mean(
    [eeg_scalp_volt_Time[i]["RNLshadow"] for i in x], axis=1
)
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1, 2, figsize=(9, 5))
plt.sca(ax[0])
# A[sig_diff_reg>0.05]=np.nan
plt.scatter(
    avTrueRNL_EEG_scalp_volt_samp,
    avShadRNL_EEG_scalp_volt_samp,
    c=np.arange(0, 18, 2),
    cmap=palette,
    vmax=18,
    vmin=0,
    label="Fixed Samples",
)
plt.scatter(
    avTrueRNL_EEG_scalp_volt_time,
    avShadRNL_EEG_scalp_volt_time,
    c=np.arange(1, 18, 2),
    cmap=palette,
    vmax=18,
    vmin=0,
    label="Fixed Time",
)
cb = plt.colorbar()
cb.ax.set_ylabel("# regions", rotation=90)
cb.ax.get_yaxis().set_ticks(np.arange(1, 18, 2), band_names)
pts = [max(plt.xlim()[0], plt.ylim()[0]), min(plt.xlim()[1], plt.ylim()[1])]
plt.xlim(plt.xlim())
plt.ylim(plt.ylim())
plt.plot(pts, pts, ":k", lw=1)
plt.xlabel("Data RNL")
plt.ylabel("Shadow RNL")
plt.legend(loc=0)

plt.sca(ax[1])
plt.scatter(
    np.arange(0, 9),
    avTrueRNL_EEG_scalp_volt_samp,
    color=plt.cm.tab20.colors[0],
    label="True",
)
plt.scatter(
    np.arange(0, 9),
    avShadRNL_EEG_scalp_volt_samp,
    color=plt.cm.tab20.colors[2],
    label="Shadow",
)
plt.scatter(
    np.arange(0, 9),
    avTrueRNL_EEG_scalp_volt_samp - avShadRNL_EEG_scalp_volt_samp,
    color=plt.cm.tab20.colors[4],
    label="Difference",
    marker="+",
)
plt.scatter(
    np.arange(0, 9),
    avTrueRNL_EEG_scalp_volt_time,
    color=plt.cm.tab20.colors[1],
    label="True",
)
plt.scatter(
    np.arange(0, 9),
    avShadRNL_EEG_scalp_volt_time,
    color=plt.cm.tab20.colors[3],
    label="Shadow",
)
plt.scatter(
    np.arange(0, 9),
    avTrueRNL_EEG_scalp_volt_time - avShadRNL_EEG_scalp_volt_time,
    color=plt.cm.tab20.colors[5],
    label="Difference",
    marker="+",
)
# plt.xscale("log")
plt.xlabel("# regions")
plt.ylabel("RNL")
plt.legend(loc=0)
plt.xticks(np.arange(0, 9), band_names)
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
plt.suptitle("Relationship between true and shadow $-$ Scalp Voltage")
plt.show()

In [ ]:
eeg_scalp_volt_Samp_NewSur = {}
for folder in glob.glob(
    "../../NonLinearityResultsNew/EEG_voltage_piecewiseSurrogates_fixedSamples_S00B?_bin10"
):
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        try:
            statsMI = np.load(os.path.join(folder, f"subject00_10.npy"))
        except:
            statsMI = np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"B(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            eeg_scalp_volt_Samp_NewSur[band] = {
                k: np.array(v) for k, v in json.load(fp).items()
            }

eeg_scalp_volt_Time_NewSur = {}
for folder in glob.glob(
    "../../NonLinearityResultsNew/trimmed_EEG_fixedTime_band?_Extended_bin*"
):
    bins = int(re.findall("bin(\d+)", folder)[0])
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        try:
            statsMI = np.load(os.path.join(folder, f"subject00_{bins}.npy"))
        except:
            statsMI = np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"band(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            eeg_scalp_volt_Time_NewSur[band] = {
                k: np.array(v) for k, v in json.load(fp).items()
            }
for i in eeg_scalp_volt_Time_NewSur:
    try:
        eeg_scalp_volt_Samp_NewSur[i]["RNL"] = (
            1
            - eeg_scalp_volt_Samp_NewSur[i]["gaussMI"]
            / eeg_scalp_volt_Samp_NewSur[i]["totalMI"]
        )
    except:
        pass
    eeg_scalp_volt_Time_NewSur[i]["RNL"] = (
        1
        - eeg_scalp_volt_Time_NewSur[i]["gaussMIshadow"]
        / eeg_scalp_volt_Time_NewSur[i]["totalMIshadow"]
    )

avShadRNL_EEG_scalp_volt_time_NewSur = []
avShadRNL_EEG_scalp_volt_samp_NewSur = []
for i in range(1, 10):
    avShadRNL_EEG_scalp_volt_time_NewSur.append(
        np.mean(eeg_scalp_volt_Time_NewSur[i]["RNL"])
        if i in eeg_scalp_volt_Time_NewSur
        else np.nan
    )
    avShadRNL_EEG_scalp_volt_samp_NewSur.append(
        np.mean(eeg_scalp_volt_Samp_NewSur[i]["RNL"])
        if i in eeg_scalp_volt_Samp_NewSur
        else np.nan
    )

In [ ]:
# tab20
avTrueRNL_EEG_scalp_volt_samp = np.mean(
    [eeg_scalp_volt_Samp[i]["RNL"] for i in x], axis=1
)
avShadRNL_EEG_scalp_volt_samp = np.mean(
    [eeg_scalp_volt_Samp[i]["RNLshadow"] for i in x], axis=1
)
avTrueRNL_EEG_scalp_volt_time = np.mean(
    [eeg_scalp_volt_Time[i]["RNL"] for i in x], axis=1
)
avShadRNL_EEG_scalp_volt_time = np.mean(
    [eeg_scalp_volt_Time[i]["RNLshadow"] for i in x], axis=1
)
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1, 2, figsize=(9, 5))
plt.sca(ax[0])
# A[sig_diff_reg>0.05]=np.nan
plt.plot(avTrueRNL_EEG_scalp_volt_samp, avShadRNL_EEG_scalp_volt_samp)
plt.scatter(
    avTrueRNL_EEG_scalp_volt_samp,
    avShadRNL_EEG_scalp_volt_samp,
    c=np.arange(0, 18, 2),
    cmap=palette,
    vmax=18,
    vmin=0,
    label="Fixed Samples",
)
plt.plot(avTrueRNL_EEG_scalp_volt_time, avShadRNL_EEG_scalp_volt_time)
plt.scatter(
    avTrueRNL_EEG_scalp_volt_time,
    avShadRNL_EEG_scalp_volt_time,
    c=np.arange(1, 18, 2),
    cmap=palette,
    vmax=18,
    vmin=0,
    label="Fixed Time",
)
plt.plot(avTrueRNL_EEG_scalp_volt_samp, avShadRNL_EEG_scalp_volt_samp_NewSur)
plt.scatter(
    avTrueRNL_EEG_scalp_volt_samp,
    avShadRNL_EEG_scalp_volt_samp_NewSur,
    c=np.arange(0, 18, 2),
    cmap=palette,
    vmax=18,
    vmin=0,
    label="Fixed Samples\nPiecewise shadow",
    marker="*",
)
plt.plot(avTrueRNL_EEG_scalp_volt_time, avShadRNL_EEG_scalp_volt_time_NewSur)
plt.scatter(
    avTrueRNL_EEG_scalp_volt_time,
    avShadRNL_EEG_scalp_volt_time_NewSur,
    c=np.arange(1, 18, 2),
    cmap=palette,
    vmax=18,
    vmin=0,
    label="Fixed Time\nExtended shadow",
    marker="*",
)
cb = plt.colorbar()
cb.ax.set_ylabel("# regions", rotation=90)
cb.ax.get_yaxis().set_ticks(np.arange(1, 18, 2), band_names)
pts = [max(plt.xlim()[0], plt.ylim()[0]), min(plt.xlim()[1], plt.ylim()[1])]
plt.xlim(plt.xlim())
plt.ylim(plt.ylim())
plt.plot(pts, pts, ":k", lw=1)
plt.xlabel("Data RNL")
plt.ylabel("Shadow RNL")
plt.legend(loc=0, frameon=False)

plt.sca(ax[1])
plt.scatter(
    np.arange(0, 9),
    avTrueRNL_EEG_scalp_volt_samp,
    color=plt.cm.tab20.colors[0],
    label="True",
)
plt.scatter(
    np.arange(0, 9),
    avShadRNL_EEG_scalp_volt_samp,
    color=plt.cm.tab20.colors[2],
    label="Shadow",
)
plt.scatter(
    np.arange(0, 9),
    avTrueRNL_EEG_scalp_volt_samp - avShadRNL_EEG_scalp_volt_samp,
    color=plt.cm.tab20.colors[4],
    label="Difference",
    marker="+",
)
plt.scatter(
    np.arange(0, 9),
    avTrueRNL_EEG_scalp_volt_time,
    color=plt.cm.tab20.colors[1],
    label="True",
)
plt.scatter(
    np.arange(0, 9),
    avShadRNL_EEG_scalp_volt_time_NewSur,
    color=plt.cm.tab20.colors[3],
    label="Shadow",
)
plt.scatter(
    np.arange(0, 9),
    avTrueRNL_EEG_scalp_volt_time - avShadRNL_EEG_scalp_volt_time_NewSur,
    color=plt.cm.tab20.colors[5],
    label="Difference",
    marker="+",
)
# plt.xscale("log")
plt.xlabel("# regions")
plt.ylabel("RNL")
ha = [
    Line2D([], [], lw=0, color=plt.cm.tab20.colors[i], marker=m)
    for i, m in zip([0, 2, 4], "oo+")
]
ha += [Line2D([], [], lw=0, color=c, marker="o") for c in ["black", "grey"]]
la = ["True", "Shadow", "Difference", "Fixed Samples", "Fixed Time"]
plt.legend(ha, la, loc=0, frameon=False)
plt.xticks(np.arange(0, 9), band_names)
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
plt.suptitle("Relationship between true and shadow $-$ Scalp Voltage")
plt.show()
pearsonr(avTrueRNL_EEG_scalp_volt_samp, avShadRNL_EEG_scalp_volt_samp), pearsonr(
    avTrueRNL_EEG_scalp_volt_time, avShadRNL_EEG_scalp_volt_time
)

In [ ]:
sr = [4, 8, 12, 15, 18, 30, 44]
plt.plot(sr, avShadRNL_EEG_scalp_volt_samp_NewSur[:7], label="samp\nPiecewise shadow")
plt.plot(sr, avShadRNL_EEG_scalp_volt_time_NewSur[:7], label="time\nExtended shadow")
plt.plot(sr, avShadRNL_EEG_scalp_volt_samp[:7], label="samp")
plt.plot(sr, avShadRNL_EEG_scalp_volt_time[:7], label="time")
plt.legend(loc=0)
plt.xlabel("Highest frequency in band")
plt.ylabel("Shadow RNL")
plt.show()

In [ ]:
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1, 2, figsize=(13, 7))
for ax_n, pack in enumerate(
    [("Samples", eeg_scalp_volt_Samp), ("Time", eeg_scalp_volt_Time)]
):
    kind, data = pack
    plt.sca(ax[ax_n])
    for i in range(1, 10):
        plt.scatter(
            data[i]["RNL"],
            data[i]["RNLshadow"],
            c=np.full(len(data[i]["RNL"]), i * 2 + (kind == "Time")),
            cmap=palette,
            vmax=18,
            vmin=0,
            s=5,
        )
    ax[ax_n].set_aspect(1)
    pts = [max(plt.xlim()[0], plt.ylim()[0]), min(plt.xlim()[1], plt.ylim()[1])]
    plt.xlim(plt.xlim())
    plt.ylim(plt.ylim())
    plt.plot(pts, pts, ":k", lw=1)
    plt.xlabel("Data RNL")
    plt.ylabel("Shadow RNL")
    plt.title(
        kind
        + (" $-$ Piecewise shadow" if kind == "Samples" else " $-$ Extended shadow")
    )
plt.show()

palette = plt.cm.magma
palette.set_bad(color="black")  # Bad values (i.e., masked, set to black 0.8
palette.set_under(color="grey")  # Bad values (i.e., masked, set to black 0.8
palette.set_over(color="grey")  # Bad values (i.e., masked, set to black 0.8
for ax_n, pack in enumerate(
    [("Samples", eeg_scalp_volt_Samp, "o"), ("Time", eeg_scalp_volt_Time, "^")]
):
    kind, data, marker = pack
    corr_eeg = [pearsonr(data[i]["RNL"], data[i]["RNLshadow"]) for i in range(1, 10)]
    plt.scatter(
        np.arange(0, 9),
        [q[0] for q in corr_eeg],
        c=[q[1] for q in corr_eeg],
        cmap=palette,
        vmax=0.05,
        vmin=0,
        marker=marker,
        label=kind
        + ("\nPiecewise shadow" if kind == "Samples" else "\nExtended shadow"),
    )
plt.xticks(np.arange(0, 9), band_names)
plt.legend(loc=0)
plt.show()

### Scalp BLP

In [ ]:
eeg_scalp_blp = {
    k: {s: [{} for b in range(9)] for s in ["Shadow", "Normal"]}
    for k in ["Time", "Samples"]
}  # eeg_scalp_blp[kind][shadow][band-1][avg]
for folder in glob.glob(
    "../../NonLinearityResultsNew/NEW_electrodeBLP_fixed*_avg*_band?_bin*"
):
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        band = int(re.findall(r"band(\d+)", folder)[0]) - 1
        avg = int(re.findall(r"avg(\d+)", folder)[0])
        kind = "Time" if "Time" in folder else "Samples"
        bins = int(re.findall(r"bin(\d+)", folder)[0])
        shadow = "Shadow" if "shadow" in folder else "Normal"
        try:
            statsMI = np.load(os.path.join(folder, f"subject00_{bins}.npy"))
        except:
            statsMI = np.array([])
        pairs = statsMI.shape[0]
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            eeg_scalp_blp[kind][shadow][band][avg] = {
                k: np.array(v) for k, v in json.load(fp).items()
            }

In [ ]:
for kind in ["Time", "Samples"]:
    for shadow in ["Shadow", "Normal"]:
        for band in range(9):
            for avg in eeg_scalp_blp[kind][shadow][band]:
                eeg_scalp_blp[kind][shadow][band][avg]["RNL"] = (
                    1
                    - eeg_scalp_blp[kind][shadow][band][avg]["gaussMI"]
                    / eeg_scalp_blp[kind][shadow][band][avg]["totalMI"]
                )
                if shadow == "Normal":
                    eeg_scalp_blp[kind][shadow][band][avg]["RNLshadow"] = (
                        1
                        - eeg_scalp_blp[kind][shadow][band][avg]["gaussMIshadow"]
                        / eeg_scalp_blp[kind][shadow][band][avg]["totalMIshadow"]
                    )

In [ ]:
colors = {"Time": "darkorange", "Samples": "b"}
STRETCH = 3
basepos = np.arange(len(band_names)) * STRETCH
fig, ax = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax_n, pack in enumerate(
    [
        ("True", "Normal", ""),
        ("Shadow After", "Normal", "shadow"),
        ("Shadow Before", "Shadow", ""),
    ]
):
    plt.sca(ax[ax_n])
    title, shadow, sub = pack
    offs = -2.5
    for avg in (1, 2, 4):
        for kind in ("Time", "Samples"):
            color = colors[kind] if avg < 4 else "magenta"
            if avg in eeg_scalp_blp[kind][shadow][0]:
                y = [
                    eeg_scalp_blp[kind][shadow][band][avg]["RNL" + sub]
                    for band in range(9)
                ]
                plt.boxplot(
                    y,
                    positions=np.arange(9) * STRETCH + offs,
                    boxprops={"color": color},
                    whiskerprops={"color": color},
                    flierprops={"markeredgecolor": color, "marker": "+"},
                    medianprops={"lw": 1.6, "color": color},
                    capprops={"color": color},
                )
                offs += 0.5

    p1 = plt.hlines(
        0.0,
        -3,
        (len(band_names) - 1) * STRETCH,
        "r",
        "solid",
        lw=1,
        label="%",
        zorder=1,
    )
    p2 = plt.hlines(
        0.1,
        -3,
        (len(band_names) - 1) * STRETCH,
        "r",
        "dashed",
        lw=1,
        label="10%",
        zorder=1,
    )
    tkpos = np.arange(9) + 1
    plt.ylabel("relative difference in MI")
    plt.xlabel("Band")
    plt.xticks(ticks=np.arange(9) * STRETCH - 1.5, labels=band_names)
    plt.title(title)
plt.legend(
    [
        Patch(facecolor="white", edgecolor="darkorange"),
        Patch(facecolor="white", edgecolor="b"),
        Patch(facecolor="white", edgecolor="magenta"),
        p1,
        p2,
    ],
    ["Fixed Time Len", "Fixed #Samples", "Both", r"0", r"10%"],
)
plt.suptitle("50 subjects $-$ Scalp BLP")
plt.show()

In [ ]:
colors = {"Time": "darkorange", "Samples": "b"}
STRETCH = 3
band_names2 = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
basepos = np.arange(len(band_names2)) * STRETCH
fig, ax = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax_n, pack in enumerate(
    [
        ("True", "Normal", ""),
        ("Shadow Before Power", "Shadow", ""),
        ("Shadow", "Normal", "shadow"),
    ]
):
    plt.sca(ax[ax_n])
    title, shadow, sub = pack
    offs = -2.5
    for avg in (1, 2, 4):
        for kind in ("Time", "Samples"):
            color = colors[kind] if avg < 4 else "magenta"
            if avg in eeg_scalp_blp[kind][shadow][0]:
                y = [
                    eeg_scalp_blp[kind][shadow][band][avg]["RNL" + sub]
                    for band in [0, 1, 2, 7, 6]
                ]
                plt.boxplot(
                    y,
                    positions=np.arange(5) * STRETCH + offs,
                    boxprops={"color": color},
                    whiskerprops={"color": color},
                    flierprops={"markeredgecolor": color, "marker": "+"},
                    medianprops={"lw": 1.6, "color": color},
                    capprops={"color": color},
                )
                offs += 0.5

    p1 = plt.hlines(
        0.0,
        -3,
        (len(band_names2) - 1) * STRETCH,
        "r",
        "solid",
        lw=1,
        label="%",
        zorder=1,
    )
    p2 = plt.hlines(
        0.1,
        -3,
        (len(band_names2) - 1) * STRETCH,
        "r",
        "dashed",
        lw=1,
        label="10%",
        zorder=1,
    )
    tkpos = np.arange(9) + 1
    plt.ylabel("relative difference in MI")
    plt.xlabel("Band")
    plt.xticks(ticks=np.arange(5) * STRETCH - 1.5, labels=band_names2)
    plt.title(title)
plt.legend(
    [
        Patch(facecolor="white", edgecolor="darkorange"),
        Patch(facecolor="white", edgecolor="b"),
        Patch(facecolor="white", edgecolor="magenta"),
        p1,
        p2,
    ],
    ["Fixed Time Len", "Fixed #Samples", "Both", r"0", r"10%"],
)
plt.suptitle("50 subjects $-$ Scalp BLP")
plt.show()

In [ ]:
STRETCH = 7
x = sorted(eeg_scalp_volt_Samp.keys())
basepos = np.arange(5) * STRETCH
fig, ax = plt.subplots(1, 1, figsize=(9, 7), sharey=True)

plt.sca(ax)
offs = -5
y = None
for avg in (1, 2, 4):
    for kind in ("Time", "Samples"):
        color = colors[kind] if avg < 4 else "magenta"
        if avg in eeg_scalp_blp[kind][shadow][0]:
            for i, (shadow, alpha) in enumerate([("Normal", 1), ("Shadow", 0.5)]):
                old_y = y
                y = [
                    eeg_scalp_blp[kind][shadow][band][avg]["RNL"]
                    for band in [0, 1, 2, 7, 6]
                ]
                plt.boxplot(
                    y,
                    positions=np.arange(5) * STRETCH + offs,
                    boxprops={"color": color, "alpha": alpha},
                    whiskerprops={"color": color, "alpha": alpha},
                    flierprops={
                        "markeredgecolor": color,
                        "alpha": alpha,
                        "marker": "+",
                    },
                    medianprops={"lw": 1.6, "color": color, "alpha": alpha},
                    capprops={"color": color, "alpha": alpha},
                )
                offs += 0.6
            sigi = np.array(
                [
                    ttest_rel(a, b, alternative="greater").pvalue
                    for a, b in zip(old_y, y)
                ]
            )
            as_sigi = np.argsort(sigi)
            HB = (
                sigi[as_sigi] < 0.05 / len(sigi) / 5
            )  # /(len(sigi)-np.arange(len(sigi)))#
            aster = as_sigi[HB]
            plt.scatter(
                (np.arange(5) * STRETCH + offs - 0.9)[aster],
                np.full_like(aster, 0.01, dtype=float),
                marker="*",
                color=(0, 0, 0),
            )

p1 = plt.hlines(
    0.0, basepos[0] - 5, basepos[-1] + 1, "r", "solid", lw=1, label="%", zorder=1
)
p3 = plt.hlines(
    0.1, basepos[0] - 5, basepos[-1] + 1, "r", "dotted", lw=1, label="10%", zorder=1
)

plt.ylabel(f"Relative Difference in MI")
plt.xlabel("Band")
band_names2 = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
plt.xticks(ticks=basepos - 2.5, labels=band_names2)
plt.legend(
    [
        Patch(facecolor="white", edgecolor="darkorange"),
        Patch(facecolor="white", edgecolor="b"),
        Patch(facecolor="white", edgecolor="magenta"),
        p1,
        p2,
        ast,
    ],
    [
        "Fixed Time Len",
        "Fixed #Samples",
        "Both",
        r"0",
        r"10%",
        "Significative\ndifference\n(Bonferroni)",
    ],
    loc=0,
)
# plt.title("50 subjects $-$ Scalp Voltage")
plt.show()

In [ ]:
# tab20
avTrueRNL_EEG_scalp_blp = {
    kind: {
        avg: np.mean(
            [eeg_scalp_blp[kind]["Normal"][band][avg]["RNL"] for band in range(9)],
            axis=1,
        )
        for avg in eeg_scalp_blp[kind]["Normal"][0]
    }
    for kind in ["Time", "Samples"]
}
avShadRNL_EEG_scalp_blp = {
    kind: {
        avg: np.mean(
            [eeg_scalp_blp[kind]["Shadow"][band][avg]["RNL"] for band in range(9)],
            axis=1,
        )
        for avg in eeg_scalp_blp[kind]["Shadow"][0]
    }
    for kind in ["Time", "Samples"]
}
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1, 2, figsize=(9, 5))
plt.sca(ax[0])
for kind in ["Samples", "Time"]:
    for av, marker in zip([1, 2, 4], "o*^"):
        if av in avTrueRNL_EEG_scalp_blp[kind]:
            plt.scatter(
                avTrueRNL_EEG_scalp_blp[kind][av],
                avShadRNL_EEG_scalp_blp[kind][av],
                c=np.arange(0, 18, 2) + (kind == "Time"),
                cmap=palette,
                vmax=18,
                vmin=0,
                marker=marker,
            )
cb = plt.colorbar()
cb.ax.set_ylabel("Band", rotation=90)
cb.ax.get_yaxis().set_ticks(np.arange(1, 18, 2), band_names)
pts = [max(plt.xlim()[0], plt.ylim()[0]), min(plt.xlim()[1], plt.ylim()[1])]
plt.xlim(plt.xlim())
plt.ylim(plt.ylim())
plt.plot(pts, pts, ":k", lw=1)
plt.xlabel("Data RNL")
plt.ylabel("Shadow RNL")
ha = [Line2D([], [], lw=0, color=c, marker="o") for c in ["black", "grey"]]
ha += [Line2D([], [], lw=0, color="k", marker=m) for m in "o*^"]
la = [
    "Fixed Samples",
    "Fixed Time",
    "125 ms",
    "250 ms",
    "500 ms",
]
plt.legend(ha, la, loc="upper left", frameon=False)

plt.sca(ax[1])
for kind in ["Samples", "Time"]:
    offset = -0.28
    for av, marker in zip([1, 2, 4], "o*^"):
        if av in avTrueRNL_EEG_scalp_blp[kind]:
            plt.scatter(
                np.arange(0, 9) + offset,
                avTrueRNL_EEG_scalp_blp[kind][av],
                color=plt.cm.tab20.colors[0 + (kind == "Time")],
                marker=marker,
            )
            plt.scatter(
                np.arange(0, 9) + offset,
                avShadRNL_EEG_scalp_blp[kind][av],
                color=plt.cm.tab20.colors[2 + (kind == "Time")],
                marker=marker,
            )
            plt.scatter(
                np.arange(0, 9) + offset,
                avTrueRNL_EEG_scalp_blp[kind][av] - avShadRNL_EEG_scalp_blp[kind][av],
                color=plt.cm.tab20.colors[4 + (kind == "Time")],
                marker=marker,
            )
        offset += 0.28
# plt.xscale("log")
plt.xlabel("Band")
plt.ylabel("RNL")
ha = [Line2D([], [], lw=0, color="k", marker=m) for m in "o*^"]
ha += [
    Line2D([], [], lw=0, color=plt.cm.tab20.colors[i], marker="o") for i in [0, 2, 4]
]
ha += [Line2D([], [], lw=0, color=c, marker="o") for c in ["black", "grey"]]
la = [
    "125 ms",
    "250 ms",
    "500 ms",
    "True",
    "Shadow",
    "Difference",
    "Fixed Samples",
    "Fixed Time",
]
plt.legend(ha, la, bbox_to_anchor=(1.2, 0.5), loc="center left", frameon=False)
plt.xticks(np.arange(0, 9), band_names)
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
plt.suptitle("Relationship between true and shadow $-$ Scalp BLP")
plt.show()

In [ ]:
# tab20
avTrueRNL_EEG_scalp_blp = {
    kind: {
        avg: np.mean(
            [eeg_scalp_blp[kind]["Normal"][band][avg]["RNL"] for band in range(9)],
            axis=1,
        )
        for avg in eeg_scalp_blp[kind]["Normal"][0]
    }
    for kind in ["Time", "Samples"]
}
avShadRNL_EEG_scalp_blp = {
    kind: {
        avg: np.mean(
            [eeg_scalp_blp[kind]["Shadow"][band][avg]["RNL"] for band in range(9)],
            axis=1,
        )
        for avg in eeg_scalp_blp[kind]["Shadow"][0]
    }
    for kind in ["Time", "Samples"]
}
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1, 2, figsize=(9, 5))
plt.sca(ax[0])
for kind in ["Samples", "Time"]:
    for av, marker in zip([1, 2, 4], "o*^"):
        if av in avTrueRNL_EEG_scalp_blp[kind]:
            plt.plot(
                avTrueRNL_EEG_scalp_blp[kind][av],
                avShadRNL_EEG_scalp_blp[kind][av],
                alpha=1 - 0.5 * (kind == "Time"),
            )
            plt.scatter(
                avTrueRNL_EEG_scalp_blp[kind][av],
                avShadRNL_EEG_scalp_blp[kind][av],
                c=np.arange(0, 18, 2) + (kind == "Time"),
                cmap=palette,
                vmax=18,
                vmin=0,
                marker=marker,
            )
cb = plt.colorbar()
cb.ax.set_ylabel("Band", rotation=90)
cb.ax.get_yaxis().set_ticks(np.arange(1, 18, 2), band_names)
pts = [max(plt.xlim()[0], plt.ylim()[0]), min(plt.xlim()[1], plt.ylim()[1])]
plt.xlim(plt.xlim())
plt.ylim(plt.ylim())
plt.plot(pts, pts, ":k", lw=1)
plt.xlabel("Data RNL")
plt.ylabel("Shadow RNL")
ha = [Line2D([], [], lw=0, color=c, marker="o") for c in ["black", "grey"]]
ha += [Line2D([], [], lw=0, color="k", marker=m) for m in "o*^"]
la = [
    "Fixed Samples",
    "Fixed Time",
    "125 ms",
    "250 ms",
    "500 ms",
]
plt.legend(ha, la, loc="upper left", frameon=False)

plt.sca(ax[1])
for kind in ["Samples", "Time"]:
    offset = -0.28
    for av, marker in zip([1, 2, 4], "o*^"):
        if av in avTrueRNL_EEG_scalp_blp[kind]:
            plt.scatter(
                np.arange(0, 9) + offset,
                avTrueRNL_EEG_scalp_blp[kind][av],
                color=plt.cm.tab20.colors[0 + (kind == "Time")],
                marker=marker,
            )
            plt.scatter(
                np.arange(0, 9) + offset,
                avShadRNL_EEG_scalp_blp[kind][av],
                color=plt.cm.tab20.colors[2 + (kind == "Time")],
                marker=marker,
            )
            plt.scatter(
                np.arange(0, 9) + offset,
                avTrueRNL_EEG_scalp_blp[kind][av] - avShadRNL_EEG_scalp_blp[kind][av],
                color=plt.cm.tab20.colors[4 + (kind == "Time")],
                marker=marker,
            )
        offset += 0.28
# plt.xscale("log")
plt.xlabel("Band")
plt.ylabel("RNL")
ha = [Line2D([], [], lw=0, color="k", marker=m) for m in "o*^"]
ha += [
    Line2D([], [], lw=0, color=plt.cm.tab20.colors[i], marker="o") for i in [0, 2, 4]
]
ha += [Line2D([], [], lw=0, color=c, marker="o") for c in ["black", "grey"]]
la = [
    "125 ms",
    "250 ms",
    "500 ms",
    "True",
    "Shadow",
    "Difference",
    "Fixed Samples",
    "Fixed Time",
]
plt.legend(ha, la, bbox_to_anchor=(1.2, 0.5), loc="center left", frameon=False)
plt.xticks(np.arange(0, 9), band_names)
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
plt.suptitle("Relationship between true and shadow $-$ Scalp BLP")
plt.show()

In [ ]:
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1, 2, figsize=(13, 7))
for ax_n, kind in enumerate(["Samples", "Time"]):
    plt.sca(ax[ax_n])
    for i in range(9):
        for avg, marker in zip(eeg_scalp_blp[kind]["Normal"][i], "o*^"):
            plt.scatter(
                eeg_scalp_blp[kind]["Normal"][i][avg]["RNL"],
                eeg_scalp_blp[kind]["Shadow"][i][avg]["RNL"],
                c=np.full(50, i * 2 + (kind == "Time")),
                cmap=palette,
                vmax=18,
                vmin=0,
                s=5,
                marker=marker,
            )
    ax[ax_n].set_aspect(1)
    pts = [max(plt.xlim()[0], plt.ylim()[0]), min(plt.xlim()[1], plt.ylim()[1])]
    plt.xlim(plt.xlim())
    plt.ylim(plt.ylim())
    plt.plot(pts, pts, ":k", lw=1)
    plt.xlabel("Data RNL")
    plt.ylabel("Shadow RNL")
    plt.title(kind)
plt.show()


palette = plt.cm.magma
palette.set_bad(color="black")  # Bad values (i.e., masked, set to black 0.8
palette.set_under(color="grey")  # Bad values (i.e., masked, set to black 0.8
palette.set_over(color="grey")  # Bad values (i.e., masked, set to black 0.8
for kind, marker in [("Samples", "o"), ("Time", "^")]:
    for avg, s in zip(eeg_scalp_blp[kind]["Normal"][0], [5, 10, 20]):
        corr_ieeg = [
            pearsonr(
                eeg_scalp_blp[kind]["Normal"][i][avg]["RNL"],
                eeg_scalp_blp[kind]["Shadow"][i][avg]["RNL"],
            )
            for i in range(9)
        ]
        plt.scatter(
            np.arange(0, 9),
            [q[0] for q in corr_ieeg],
            c=[q[1] for q in corr_ieeg],
            cmap=palette,
            vmax=0.05,
            vmin=0,
            marker=marker,
            label=kind if avg == 2 else None,
            s=s,
        )
plt.xticks(np.arange(0, 9), band_names)
plt.legend(loc=0)
plt.show()

### Source BLP

In [ ]:
eeg_source_blp = {
    k: {s: [{} for b in range(9)] for s in ["Shadow", "Normal"]}
    for k in ["Time", "Samples"]
}  # eeg_source_blp[kind][shadow][band-1][avg]
for folder in glob.glob(
    "../../NonLinearityResultsNew/PCA_sourceBLP_fixed*_avg*_band?_bin*"
):
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        band = int(re.findall(r"band(\d+)", folder)[0]) - 1
        avg = int(re.findall(r"avg(\d+)", folder)[0])
        kind = "Time" if "Time" in folder else "Samples"
        bins = int(re.findall(r"bin(\d+)", folder)[0])
        shadow = "Shadow" if "shadow" in folder else "Normal"
        try:
            assert False
            statsMI = np.load(os.path.join(folder, f"subject00_{bins}.npy"))
        except:
            statsMI = np.array([])
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            eeg_source_blp[kind][shadow][band][avg] = {
                k: np.array(v) for k, v in json.load(fp).items()
            }

In [ ]:
for kind in ["Time", "Samples"]:
    for shadow in ["Shadow", "Normal"]:
        for band in range(9):
            for avg in eeg_source_blp[kind][shadow][band]:
                eeg_source_blp[kind][shadow][band][avg]["RNL"] = (
                    1
                    - eeg_source_blp[kind][shadow][band][avg]["gaussMI"]
                    / eeg_source_blp[kind][shadow][band][avg]["totalMI"]
                )
                if shadow == "Normal":
                    eeg_source_blp[kind][shadow][band][avg]["RNLshadow"] = (
                        1
                        - eeg_source_blp[kind][shadow][band][avg]["gaussMIshadow"]
                        / eeg_source_blp[kind][shadow][band][avg]["totalMIshadow"]
                    )

In [ ]:
colors = {"Time": "darkorange", "Samples": "b"}
STRETCH = 3
basepos = np.arange(len(band_names)) * STRETCH
fig, ax = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax_n, pack in enumerate(
    [
        ("True", "Normal", ""),
        ("Shadow After", "Normal", "shadow"),
        ("Shadow Before", "Shadow", ""),
    ]
):
    plt.sca(ax[ax_n])
    title, shadow, sub = pack
    offs = -2.5
    for avg in (1, 2, 4):
        for kind in ("Time", "Samples"):
            color = colors[kind] if avg < 4 else "magenta"
            if avg in eeg_source_blp[kind][shadow][0]:
                y = [
                    eeg_source_blp[kind][shadow][band][avg]["RNL" + sub]
                    for band in range(9)
                ]
                plt.boxplot(
                    y,
                    positions=np.arange(9) * STRETCH + offs,
                    boxprops={"color": color},
                    whiskerprops={"color": color},
                    flierprops={"markeredgecolor": color, "marker": "+"},
                    medianprops={"lw": 1.6, "color": color},
                    capprops={"color": color},
                )
                offs += 0.5

    p1 = plt.hlines(
        0.0,
        -3,
        (len(band_names) - 1) * STRETCH,
        "r",
        "solid",
        lw=1,
        label="%",
        zorder=1,
    )
    p2 = plt.hlines(
        0.1,
        -3,
        (len(band_names) - 1) * STRETCH,
        "r",
        "dashed",
        lw=1,
        label="10%",
        zorder=1,
    )
    tkpos = np.arange(9) + 1
    plt.ylabel("relative difference in MI")
    plt.xlabel("Band")
    plt.xticks(ticks=np.arange(9) * STRETCH - 1.5, labels=band_names)
    plt.title(title)
plt.legend(
    [
        Patch(facecolor="white", edgecolor="darkorange"),
        Patch(facecolor="white", edgecolor="b"),
        Patch(facecolor="white", edgecolor="magenta"),
        p1,
        p2,
    ],
    ["Fixed Time Len", "Fixed #Samples", "Both", r"0", r"10%"],
)
plt.suptitle("50 subjects $-$ Source BLP")
plt.show()

In [ ]:
STRETCH = 7
x = sorted(eeg_scalp_volt_Samp.keys())
basepos = np.arange(5) * STRETCH
fig, ax = plt.subplots(1, 1, figsize=(9, 7), sharey=True)

plt.sca(ax)
offs = -5
y = None
for avg in (1, 2, 4):
    for kind in ("Time", "Samples"):
        color = colors[kind] if avg < 4 else "magenta"
        if avg in eeg_source_blp[kind][shadow][0]:
            for i, (shadow, alpha) in enumerate([("Normal", 1), ("Shadow", 0.5)]):
                old_y = y
                y = [
                    eeg_source_blp[kind][shadow][band][avg]["RNL"]
                    for band in [0, 1, 2, 7, 6]
                ]
                plt.boxplot(
                    y,
                    positions=np.arange(5) * STRETCH + offs,
                    boxprops={"color": color, "alpha": alpha},
                    whiskerprops={"color": color, "alpha": alpha},
                    flierprops={
                        "markeredgecolor": color,
                        "alpha": alpha,
                        "marker": "+",
                    },
                    medianprops={"lw": 1.6, "color": color, "alpha": alpha},
                    capprops={"color": color, "alpha": alpha},
                )
                offs += 0.6
            sigi = np.array(
                [
                    ttest_rel(a, b, alternative="greater").pvalue
                    for a, b in zip(old_y, y)
                ]
            )
            as_sigi = np.argsort(sigi)
            HB = (
                sigi[as_sigi] < 0.05 / len(sigi) / 5
            )  # /(len(sigi)-np.arange(len(sigi)))#
            aster = as_sigi[HB]
            plt.scatter(
                (np.arange(5) * STRETCH + offs - 0.9)[aster],
                np.full_like(aster, 0.01, dtype=float),
                marker="*",
                color=(0, 0, 0),
            )

p1 = plt.hlines(
    0.0, basepos[0] - 5, basepos[-1] + 1, "r", "solid", lw=1, label="%", zorder=1
)
p3 = plt.hlines(
    0.1, basepos[0] - 5, basepos[-1] + 1, "r", "dotted", lw=1, label="10%", zorder=1
)

plt.ylabel(f"Relative Difference in MI")
plt.xlabel("Band")
band_names2 = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
plt.xticks(ticks=basepos - 2.5, labels=band_names2)
plt.legend(
    [
        Patch(facecolor="white", edgecolor="darkorange"),
        Patch(facecolor="white", edgecolor="b"),
        Patch(facecolor="white", edgecolor="magenta"),
        p1,
        p2,
        ast,
    ],
    [
        "Fixed Time Len",
        "Fixed #Samples",
        "Both",
        r"0",
        r"10%",
        "Significative\ndifference\n(Bonferroni)",
    ],
    loc=0,
)
# plt.title("50 subjects $-$ Scalp Voltage")
plt.title("Source BLP")
plt.show()

In [ ]:
# tab20
avTrueRNL_EEG_source_blp = {
    kind: {
        avg: np.mean(
            [eeg_source_blp[kind]["Normal"][band][avg]["RNL"] for band in range(9)],
            axis=1,
        )
        for avg in eeg_source_blp[kind]["Normal"][0]
    }
    for kind in ["Time", "Samples"]
}
avShadRNL_EEG_source_blp = {
    kind: {
        avg: np.mean(
            [eeg_source_blp[kind]["Shadow"][band][avg]["RNL"] for band in range(9)],
            axis=1,
        )
        for avg in eeg_source_blp[kind]["Shadow"][0]
    }
    for kind in ["Time", "Samples"]
}
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1, 2, figsize=(9, 5))
plt.sca(ax[0])
for kind in ["Samples", "Time"]:
    for av, marker in zip([1, 2, 4], "o*^"):
        if av in avTrueRNL_EEG_source_blp[kind]:
            plt.scatter(
                avTrueRNL_EEG_source_blp[kind][av],
                avShadRNL_EEG_source_blp[kind][av],
                c=np.arange(0, 18, 2) + (kind == "Time"),
                cmap=palette,
                vmax=18,
                vmin=0,
                marker=marker,
            )
cb = plt.colorbar()
cb.ax.set_ylabel("Band", rotation=90)
cb.ax.get_yaxis().set_ticks(np.arange(1, 18, 2), band_names)
pts = [max(plt.xlim()[0], plt.ylim()[0]), min(plt.xlim()[1], plt.ylim()[1])]
plt.xlim(plt.xlim())
plt.ylim(plt.ylim())
plt.plot(pts, pts, ":k", lw=1)
plt.xlabel("Data RNL")
plt.ylabel("Shadow RNL")
ha = [Line2D([], [], lw=0, color=c, marker="o") for c in ["black", "grey"]]
ha += [Line2D([], [], lw=0, color="k", marker=m) for m in "o*^"]
la = [
    "Fixed Samples",
    "Fixed Time",
    "125 ms",
    "250 ms",
    "500 ms",
]
plt.legend(ha, la, loc="upper left", frameon=False)

plt.sca(ax[1])
for kind in ["Samples", "Time"]:
    offset = -0.28
    for av, marker in zip([1, 2, 4], "o*^"):
        if av in avTrueRNL_EEG_source_blp[kind]:
            plt.scatter(
                np.arange(0, 9) + offset,
                avTrueRNL_EEG_source_blp[kind][av],
                color=plt.cm.tab20.colors[0 + (kind == "Time")],
                marker=marker,
            )
            plt.scatter(
                np.arange(0, 9) + offset,
                avShadRNL_EEG_source_blp[kind][av],
                color=plt.cm.tab20.colors[2 + (kind == "Time")],
                marker=marker,
            )
            plt.scatter(
                np.arange(0, 9) + offset,
                avTrueRNL_EEG_source_blp[kind][av] - avShadRNL_EEG_source_blp[kind][av],
                color=plt.cm.tab20.colors[4 + (kind == "Time")],
                marker=marker,
            )
        offset += 0.28
# plt.xscale("log")
plt.xlabel("Band")
plt.ylabel("RNL")
ha = [Line2D([], [], lw=0, color="k", marker=m) for m in "o*^"]
ha += [
    Line2D([], [], lw=0, color=plt.cm.tab20.colors[i], marker="o") for i in [0, 2, 4]
]
ha += [Line2D([], [], lw=0, color=c, marker="o") for c in ["black", "grey"]]
la = [
    "125 ms",
    "250 ms",
    "500 ms",
    "True",
    "Shadow",
    "Difference",
    "Fixed Samples",
    "Fixed Time",
]
plt.legend(ha, la, bbox_to_anchor=(1.2, 0.5), loc="center left", frameon=False)
plt.xticks(np.arange(0, 9), band_names)
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
plt.suptitle("Relationship between true and shadow $-$ Source BLP")
plt.show()

In [ ]:
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1, 2, figsize=(13, 7))
for ax_n, kind in enumerate(["Samples", "Time"]):
    plt.sca(ax[ax_n])
    for i in range(9):
        for avg, marker in zip(eeg_source_blp[kind]["Normal"][i], "o*^"):
            plt.scatter(
                eeg_source_blp[kind]["Normal"][i][avg]["RNL"],
                eeg_source_blp[kind]["Shadow"][i][avg]["RNL"],
                c=np.full(50, i * 2 + (kind == "Time")),
                cmap=palette,
                vmax=18,
                vmin=0,
                s=5,
                marker=marker,
            )
    ax[ax_n].set_aspect(1)
    pts = [max(plt.xlim()[0], plt.ylim()[0]), min(plt.xlim()[1], plt.ylim()[1])]
    plt.xlim(plt.xlim())
    plt.ylim(plt.ylim())
    plt.plot(pts, pts, ":k", lw=1)
    plt.xlabel("Data RNL")
    plt.ylabel("Shadow RNL")
    plt.title(kind)
plt.show()


palette = plt.cm.magma
palette.set_bad(color="black")  # Bad values (i.e., masked, set to black 0.8
palette.set_under(color="grey")  # Bad values (i.e., masked, set to black 0.8
palette.set_over(color="grey")  # Bad values (i.e., masked, set to black 0.8
for kind, marker in [("Samples", "o"), ("Time", "^")]:
    for avg, s in zip(eeg_source_blp[kind]["Normal"][0], [5, 10, 20]):
        corr_ieeg = [
            pearsonr(
                eeg_source_blp[kind]["Normal"][i][avg]["RNL"],
                eeg_source_blp[kind]["Shadow"][i][avg]["RNL"],
            )
            for i in range(9)
        ]
        plt.scatter(
            np.arange(0, 9),
            [q[0] for q in corr_ieeg],
            c=[q[1] for q in corr_ieeg],
            cmap=palette,
            vmax=0.05,
            vmin=0,
            marker=marker,
            label=kind if avg == 2 else None,
            s=s,
        )
plt.xticks(np.arange(0, 9), band_names)
plt.legend(loc=0)
plt.show()

In [ ]:
for kind in ["Samples", "Time"]:
    for band in range(9):
        for avg in eeg_source_blp[kind]["Normal"][band]:
            print(
                kind,
                band_names[band][2:9],
                125 * avg,
                pearsonr(
                    eeg_source_blp[kind]["Normal"][band][avg]["RNL"],
                    eeg_scalp_blp[kind]["Normal"][band][avg]["RNL"],
                )[0],
                sep="\t",
            )

## iEEG

**Fixed time** corresponds to *45 s*

**Fixed samples** is *1000 samples*

In particular $\delta$ and $\theta$ fixed time bands have less than 1000 samples (405, 810).

In [ ]:
for b in range(1, 10):
    mat = loadmat(f"../../NonLinearityData/iEEG/FIX_iEEG_fixedTime_band{b}.mat")["EEG"]
    display(Latex(band_names[b - 1] + f" = {mat.shape[0]}"))

In [ ]:
import glob, re, os, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel
from matplotlib.patches import Patch

sns.set_theme("talk", "ticks")

In [ ]:
ieeg_volt = {}
for folder in glob.glob("/home/raffaelli/NonLinearity/NonLinearityResultsNew/iEEG_12276_band?_MF_bin*"):
    bins = int(re.findall("bin(\d+)", folder)[0])
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        try:
            statsMI = np.load(os.path.join(folder, f"subject00_{bins}.npy"))
        except:
            statsMI = np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"band(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            ieeg_volt[band] = {k: np.array(v) for k, v in json.load(fp).items()}
for i in ieeg_volt:
    ieeg_volt[i]["RNL"] = (
        ieeg_volt[i]["totalMI"] - ieeg_volt[i]["gaussMI"]
    ) / ieeg_volt[i]["totalMI"]
    ieeg_volt[i]["RNLshadow"] = (
        ieeg_volt[i]["totalMIshadow"] - ieeg_volt[i]["gaussMIshadow"]
    ) / ieeg_volt[i]["totalMIshadow"]
    
ieeg_despiked = {}
for folder in glob.glob("/home/raffaelli/NonLinearity/NonLinearityResultsNew/iEEG_despiked_band?_bin*"):
    bins = int(re.findall("bin(\d+)", folder)[0])
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        try:
            statsMI = np.load(os.path.join(folder, f"subject00_{bins}.npy"))
        except:
            statsMI = np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"band(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            ieeg_despiked[band] = {k: np.array(v) for k, v in json.load(fp).items()}
for i in ieeg_despiked:
    ieeg_despiked[i]["RNL"] = (
        ieeg_despiked[i]["totalMI"] - ieeg_despiked[i]["gaussMI"]
    ) / ieeg_despiked[i]["totalMI"]
    ieeg_despiked[i]["RNLshadow"] = (
        ieeg_despiked[i]["totalMIshadow"] - ieeg_despiked[i]["gaussMIshadow"]
    ) / ieeg_despiked[i]["totalMIshadow"]

ieeg_volt_shad = {}
for folder in glob.glob("/home/raffaelli/NonLinearity/NonLinearityResultsNew/iEEG_band5__asBand?_bin*"):
    bins = int(re.findall("bin(\d+)", folder)[0])
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        try:
            statsMI = np.load(os.path.join(folder, f"subject00_{bins}.npy"))
        except:
            statsMI = np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"asBand(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            ieeg_volt_shad[band] = {k: np.array(v) for k, v in json.load(fp).items()}
for i in ieeg_volt_shad:
    ieeg_volt_shad[i]["RNLshadow"] = (
        ieeg_volt_shad[i]["totalMIshadow"] - ieeg_volt_shad[i]["gaussMIshadow"]
    ) / ieeg_volt_shad[i]["totalMIshadow"]

In [ ]:
STRETCH = 1.5
x = sorted(ieeg_volt.keys())
basepos = np.arange(5) * STRETCH
fig, ax = plt.subplots(1, 1, figsize=(9, 7), sharey=True)

plt.sca(ax)
pos = basepos - 0.33
y = None
for i, (who, color) in enumerate([(ieeg_volt, "darkorange"), (ieeg_volt_shad, "blue")]):
    old_y = y
    y = [who[i]["RNLshadow"] for i in [1, 2, 3, 4, 5]]
    plt.boxplot(
        y,
        positions=pos,
        widths=0.6,
        boxprops={"color": color},
        whiskerprops={"color": color},
        flierprops={"markeredgecolor": color, "marker": "+"},
        medianprops={"lw": 1.6, "color": color},
        capprops={"color": color},
    )
    pos = basepos + 0.33

p1 = plt.hlines(
    0.0, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "solid", lw=1, label="%", zorder=1
)
p2 = plt.hlines(
    0.01, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "dashed", lw=1, label="1%", zorder=1
)
p3 = plt.hlines(
    0.1, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "dotted", lw=1, label="10%", zorder=1
)

sigi = np.array(
    [ttest_rel(a, b, alternative="greater").pvalue for a, b in zip(old_y, y)]
)
as_sigi = np.argsort(sigi)
HB = sigi[as_sigi] < 0.05 / len(sigi)  # /(len(sigi)-np.arange(len(sigi)))#
aster = as_sigi[HB]
plt.scatter(
    (np.arange(5) * STRETCH)[aster],
    np.full_like(aster, 0.2, dtype=float),
    marker="*",
    color=(0, 0, 0),
)

plt.ylabel(f"Relative Difference in MI")
plt.xlabel("Band")
band_names2 = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
plt.xticks(ticks=basepos, labels=band_names2)
plt.legend(
    [
        Patch(facecolor="white", edgecolor="darkorange"),
        Patch(facecolor="white", edgecolor="blue"),
        p1,
        p2,
        p3,
        aster,
    ],
    ["Original\nshadow", "Truncated\nshadow $\\gamma$", r"0%", r"1%", r"10%", "Significative\ndifference\n(Bonferroni)"],
    loc=0,
)
# plt.title("50 subjects $-$ Scalp Voltage")
plt.show()

In [ ]:
STRETCH = 1.5
x = sorted(ieeg_volt.keys())
basepos = np.arange(5) * STRETCH
fig, ax = plt.subplots(1, 1, figsize=(9, 7), sharey=True)

plt.sca(ax)
pos = basepos - 0.33
y = None
for i, (who, color) in enumerate([(ieeg_volt, "darkorange"), (ieeg_despiked, "blue")]):
    old_y = y
    y = [who[i]["RNL"] for i in [1, 2, 3, 4, 5]]
    plt.boxplot(
        y,
        positions=pos,
        widths=0.6,
        boxprops={"color": color},
        whiskerprops={"color": color},
        flierprops={"markeredgecolor": color, "marker": "+"},
        medianprops={"lw": 1.6, "color": color},
        capprops={"color": color},
    )
    pos = basepos + 0.33

p1 = plt.hlines(
    0.0, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "solid", lw=1, label="%", zorder=1
)
p2 = plt.hlines(
    0.01, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "dashed", lw=1, label="1%", zorder=1
)
p3 = plt.hlines(
    0.1, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "dotted", lw=1, label="10%", zorder=1
)

sigi = np.array(
    [ttest_rel(a, b, alternative="greater").pvalue for a, b in zip(old_y, y)]
)
as_sigi = np.argsort(sigi)
HB = sigi[as_sigi] < 0.05 / len(sigi)  # /(len(sigi)-np.arange(len(sigi)))#
aster = as_sigi[HB]
plt.scatter(
    (np.arange(5) * STRETCH)[aster],
    np.full_like(aster, 0.2, dtype=float),
    marker="*",
    color=(0, 0, 0),
)

plt.ylabel(f"Relative Difference in MI")
plt.xlabel("Band")
band_names2 = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
plt.xticks(ticks=basepos, labels=band_names2)
plt.legend(
    [
        Patch(facecolor="white", edgecolor="darkorange"),
        Patch(facecolor="white", edgecolor="blue"),
        p1,
        p2,
        p3,
        aster,
    ],
    ["With IED", "Without IED", r"0%", r"1%", r"10%", "Significative\ndifference\n(Bonferroni)"],
    loc=0,
)
# plt.title("50 subjects $-$ Scalp Voltage")
plt.show()

In [ ]:
STRETCH = 1.5
x = sorted(ieeg_volt.keys())
basepos = np.arange(5) * STRETCH
fig, ax = plt.subplots(1, 1, figsize=(9, 7), sharey=True)

plt.sca(ax)
pos = basepos - 0.33
y = None
for i, (extra, color) in enumerate([("", "darkorange"), ("shadow", "blue")]):
    old_y = y
    y = [ieeg_volt[i]["RNL" + extra] for i in [1, 2, 3, 4, 5]]
    plt.boxplot(
        y,
        positions=pos,
        widths=0.6,
        boxprops={"color": color},
        whiskerprops={"color": color},
        flierprops={"markeredgecolor": color, "marker": "+"},
        medianprops={"lw": 1.6, "color": color},
        capprops={"color": color},
    )
    pos = basepos + 0.33

p1 = plt.hlines(
    0.0, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "solid", lw=1, label="%", zorder=1
)
p2 = plt.hlines(
    0.01, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "dashed", lw=1, label="1%", zorder=1
)
p3 = plt.hlines(
    0.1, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "dotted", lw=1, label="10%", zorder=1
)

sigi = np.array(
    [ttest_rel(a, b, alternative="greater").pvalue for a, b in zip(old_y, y)]
)
as_sigi = np.argsort(sigi)
HB = sigi[as_sigi] < 0.05 / len(sigi)  # /(len(sigi)-np.arange(len(sigi)))#
aster = as_sigi[HB]
plt.scatter(
    (np.arange(5) * STRETCH)[aster],
    np.full_like(aster, 0.2, dtype=float),
    marker="*",
    color=(0, 0, 0),
)

plt.ylabel(f"Relative Difference in MI")
plt.xlabel("Band")
band_names2 = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
plt.xticks(ticks=basepos, labels=band_names2)
plt.legend(
    [
        Patch(facecolor="white", edgecolor="darkorange"),
        Patch(facecolor="white", edgecolor="blue"),
        p1,
        p2,
        p3,
        aster,
    ],
    ["True", "Shadow", r"0%", r"1%", r"10%", "Significative\ndifference\n(Bonferroni)"],
    loc=0,
)
# plt.title("50 subjects $-$ Scalp Voltage")
plt.show()

In [ ]:
import pandas as pd
spikes = pd.read_csv("../../NonLinearityData/iEEG/iEEG_info.csv", index_col=0)
spikes

In [ ]:
from scipy.stats import pearsonr
for band in range(1,6):
    r = pearsonr(spikes.Tot_spikes, ieeg_volt[band]["RNL"])
    sig = "*" if r.pvalue<0.01 else ""
    plt.scatter(spikes.Tot_spikes, ieeg_volt[band]["RNL"], label=f"{band_names2[band-1]} (r: {r.statistic:.3} {sig})")
plt.xlabel("# spikes")
plt.ylabel("RNL")
plt.legend(bbox_to_anchor=(1., 0.5), loc="center left", title="Band")
plt.title("Relationship betwee IED and RNL")
plt.show()

In [ ]:

filt = np.ones(18, dtype=bool)
# filt[4]=False
x = spikes.Tot_spikes[filt]
for band in range(1,6):
    y = ieeg_volt[band]["RNL"][filt]
    print(f"Band {band} - {band_names2[band-1]}", pearsonr(x, y), pearsonr(x[x<400], y[x<400]))

In [ ]:
ieeg_volt_Samp = {}
for folder in glob.glob(
    "../../NonLinearityResultsNew/FIX_iEEG_fixedSamples_band?_bin10"
):
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        try:
            statsMI = np.load(os.path.join(folder, f"subject00_10.npy"))
        except:
            statsMI = np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"band(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            ieeg_volt_Samp[band] = {k: np.array(v) for k, v in json.load(fp).items()}

ieeg_volt_Time = {}
for folder in glob.glob("../../NonLinearityResultsNew/FIX_iEEG_fixedTime_band?_bin*"):
    bins = int(re.findall("bin(\d+)", folder)[0])
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        try:
            statsMI = np.load(os.path.join(folder, f"subject00_{bins}.npy"))
        except:
            statsMI = np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"band(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            ieeg_volt_Time[band] = {k: np.array(v) for k, v in json.load(fp).items()}

In [ ]:
for i in ieeg_volt_Samp:
    ieeg_volt_Samp[i]["RNL"] = (
        ieeg_volt_Samp[i]["totalMI"] - ieeg_volt_Samp[i]["gaussMI"]
    ) / ieeg_volt_Samp[i]["totalMI"]
    ieeg_volt_Samp[i]["RNLshadow"] = (
        ieeg_volt_Samp[i]["totalMIshadow"] - ieeg_volt_Samp[i]["gaussMIshadow"]
    ) / ieeg_volt_Samp[i]["totalMIshadow"]
    ieeg_volt_Time[i]["RNL"] = (
        ieeg_volt_Time[i]["totalMI"] - ieeg_volt_Time[i]["gaussMI"]
    ) / ieeg_volt_Time[i]["totalMI"]
    ieeg_volt_Time[i]["RNLshadow"] = (
        ieeg_volt_Time[i]["totalMIshadow"] - ieeg_volt_Time[i]["gaussMIshadow"]
    ) / ieeg_volt_Time[i]["totalMIshadow"]

In [ ]:
STRETCH = 1.5
x = sorted(ieeg_volt_Samp.keys())
basepos = np.arange(len(x)) * STRETCH
fig, ax = plt.subplots(1, 2, figsize=(9, 5), sharey=True)

for i, extra in enumerate(["", "shadow"]):
    plt.sca(ax[i])
    pos = basepos - 0.33
    y = [ieeg_volt_Time[i]["RNL" + extra] for i in x]
    plt.boxplot(
        y,
        positions=pos,
        widths=0.6,
        boxprops={"color": "blue"},
        whiskerprops={"color": "blue"},
        flierprops={"markeredgecolor": "blue", "marker": "+"},
        medianprops={"lw": 1.6, "color": "blue"},
        capprops={"color": "blue"},
    )

    pos = basepos + 0.33
    y = [ieeg_volt_Samp[i]["RNL" + extra] for i in x]
    plt.boxplot(
        y,
        positions=pos,
        widths=0.6,
        boxprops={"color": "darkorange"},
        whiskerprops={"color": "darkorange"},
        flierprops={"markeredgecolor": "darkorange", "marker": "+"},
        medianprops={"lw": 1.6, "color": "darkorange"},
        capprops={"color": "darkorange"},
    )

    p1 = plt.hlines(
        0.0,
        basepos[0] - 0.6,
        basepos[-1] + 0.6,
        "r",
        "solid",
        lw=1,
        label="%",
        zorder=1,
    )
    p2 = plt.hlines(
        0.1,
        basepos[0] - 0.6,
        basepos[-1] + 0.6,
        "r",
        "dotted",
        lw=1,
        label="10%",
        zorder=1,
    )

    plt.ylabel(f"relative difference in {extra} MI")
    plt.xlabel("Band")
    plt.xticks(ticks=basepos, labels=band_names)
    plt.title("Shadow" if extra else "True")
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
plt.legend(
    [
        Patch(facecolor="white", edgecolor="blue"),
        Patch(facecolor="white", edgecolor="darkorange"),
        p1,
        p2,
    ],
    ["Fixed time", "Fixed #samples", r"0", r"10%"],
    loc=0,
)
plt.suptitle("18 subjects $-$ Stereo Voltage")
plt.show()

In [ ]:
STRETCH = 1.5
x = sorted(ieeg_volt_Samp.keys())
basepos = np.arange(5) * STRETCH
fig, ax = plt.subplots(1, 1, figsize=(9, 7), sharey=True)

plt.sca(ax)
pos = basepos - 0.33
y = None
for i, (extra, color) in enumerate([("", "darkorange"), ("shadow", "blue")]):
    old_y = y
    y = [ieeg_volt_Samp[i]["RNL" + extra] for i in [1, 2, 3, 8, 7]]
    plt.boxplot(
        y,
        positions=pos,
        widths=0.6,
        boxprops={"color": color},
        whiskerprops={"color": color},
        flierprops={"markeredgecolor": color, "marker": "+"},
        medianprops={"lw": 1.6, "color": color},
        capprops={"color": color},
    )
    pos = basepos + 0.33

p1 = plt.hlines(
    0.0, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "solid", lw=1, label="%", zorder=1
)
p2 = plt.hlines(
    0.01, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "dashed", lw=1, label="1%", zorder=1
)
p3 = plt.hlines(
    0.1, basepos[0] - 0.6, basepos[-1] + 0.6, "r", "dotted", lw=1, label="10%", zorder=1
)

sigi = np.array(
    [ttest_rel(a, b, alternative="greater").pvalue for a, b in zip(old_y, y)]
)
as_sigi = np.argsort(sigi)
HB = sigi[as_sigi] < 0.05 / len(sigi)  # /(len(sigi)-np.arange(len(sigi)))#
aster = as_sigi[HB]
plt.scatter(
    (np.arange(5) * STRETCH + offs - 0.9)[aster],
    np.full_like(aster, 0.2, dtype=float),
    marker="*",
    color=(0, 0, 0),
)

plt.ylabel(f"Relative Difference in MI")
plt.xlabel("Band")
band_names2 = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
plt.xticks(ticks=basepos, labels=band_names2)
plt.legend(
    [
        Patch(facecolor="white", edgecolor="darkorange"),
        Patch(facecolor="white", edgecolor="blue"),
        p1,
        p2,
        p3,
        ast,
    ],
    ["True", "Shadow", r"0%", r"1%", r"10%", "Significative\ndifference\n(Bonferroni)"],
    loc=0,
)
# plt.title("50 subjects $-$ Scalp Voltage")
plt.show()

In [ ]:
# tab20
avTrueRNL_iEEG_volt_samp = np.mean([ieeg_volt_Samp[i]["RNL"] for i in x], axis=1)
avShadRNL_iEEG_volt_samp = np.mean([ieeg_volt_Samp[i]["RNLshadow"] for i in x], axis=1)
avTrueRNL_iEEG_volt_time = np.mean([ieeg_volt_Time[i]["RNL"] for i in x], axis=1)
avShadRNL_iEEG_volt_time = np.mean([ieeg_volt_Time[i]["RNLshadow"] for i in x], axis=1)
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1, 2, figsize=(9, 5))
plt.sca(ax[0])
# A[sig_diff_reg>0.05]=np.nan
plt.scatter(
    avTrueRNL_iEEG_volt_samp,
    avShadRNL_iEEG_volt_samp,
    c=np.arange(0, 18, 2),
    cmap=palette,
    vmax=18,
    vmin=0,
    label="Fixed Samples",
)
plt.scatter(
    avTrueRNL_iEEG_volt_time,
    avShadRNL_iEEG_volt_time,
    c=np.arange(1, 18, 2),
    cmap=palette,
    vmax=18,
    vmin=0,
    label="Fixed Time",
)
cb = plt.colorbar()
cb.ax.set_ylabel("# regions", rotation=90)
cb.ax.get_yaxis().set_ticks(np.arange(1, 18, 2), band_names)
pts = [max(plt.xlim()[0], plt.ylim()[0]), min(plt.xlim()[1], plt.ylim()[1])]
plt.xlim(plt.xlim())
plt.ylim(plt.ylim())
plt.plot(pts, pts, ":k", lw=1)
plt.xlabel("Data RNL")
plt.ylabel("Shadow RNL")
plt.legend(loc=0, frameon=False)

plt.sca(ax[1])
plt.scatter(
    np.arange(0, 9),
    avTrueRNL_iEEG_volt_samp,
    color=plt.cm.tab20.colors[0],
    label="True",
)
plt.scatter(
    np.arange(0, 9),
    avShadRNL_iEEG_volt_samp,
    color=plt.cm.tab20.colors[2],
    label="Shadow",
)
plt.scatter(
    np.arange(0, 9),
    avTrueRNL_iEEG_volt_samp - avShadRNL_iEEG_volt_samp,
    color=plt.cm.tab20.colors[4],
    label="Difference",
    marker="+",
)
plt.scatter(
    np.arange(0, 9),
    avTrueRNL_iEEG_volt_time,
    color=plt.cm.tab20.colors[1],
    label="True",
)
plt.scatter(
    np.arange(0, 9),
    avShadRNL_iEEG_volt_time,
    color=plt.cm.tab20.colors[3],
    label="Shadow",
)
plt.scatter(
    np.arange(0, 9),
    avTrueRNL_iEEG_volt_time - avShadRNL_iEEG_volt_time,
    color=plt.cm.tab20.colors[5],
    label="Difference",
    marker="+",
)
# plt.xscale("log")
plt.xlabel("# regions")
plt.ylabel("RNL")
plt.legend(bbox_to_anchor=(1.2, 0.5), loc="center left", frameon=False)
plt.xticks(np.arange(0, 9), band_names)
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
plt.suptitle("Relationship between true and shadow $-$ Stereo Voltage")
plt.show()

In [ ]:
iEEG_volt_TimeTrue_NewSur = {}
for folder in glob.glob(
    "../../NonLinearityResultsNew/iEEG_voltage_piecewiseSurrogates_fixedTime_T00B?_bin*"
):
    bins = int(re.findall("bin(\d+)", folder)[0])
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        try:
            statsMI = np.load(os.path.join(folder, f"subject00_{bins}.npy"))
        except:
            statsMI = np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"B(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            iEEG_volt_TimeTrue_NewSur[band] = {
                k: np.array(v) for k, v in json.load(fp).items()
            }

iEEG_volt_Time_NewSur = {}
for folder in glob.glob(
    "../../NonLinearityResultsNew/iEEG_voltage_piecewiseSurrogates_fixedTime_S00B?_bin*"
):
    bins = int(re.findall("bin(\d+)", folder)[0])
    if os.path.isfile(
        os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
    ):
        try:
            statsMI = np.load(os.path.join(folder, f"subject00_{bins}.npy"))
        except:
            statsMI = np.array([])
        pairs = statsMI.shape[0]
        band = int(re.findall(r"B(\d+)", folder)[0])
        print(folder, band, statsMI.shape)
        with open(
            os.path.join(folder, os.path.split(folder)[1] + "_globalStats.json")
        ) as fp:
            iEEG_volt_Time_NewSur[band] = {
                k: np.array(v) for k, v in json.load(fp).items()
            }
for i in iEEG_volt_TimeTrue_NewSur:
    iEEG_volt_TimeTrue_NewSur[i]["RNL"] = (
        1
        - iEEG_volt_TimeTrue_NewSur[i]["gaussMI"]
        / iEEG_volt_TimeTrue_NewSur[i]["totalMI"]
    )
    iEEG_volt_Time_NewSur[i]["RNL"] = (
        1 - iEEG_volt_Time_NewSur[i]["gaussMI"] / iEEG_volt_Time_NewSur[i]["totalMI"]
    )

avShadRNL_iEEG_volt_time_NewSur = []
avShadRNL_iEEG_volt_timeTrue_NewSur = []
for i in range(1, 10):
    avShadRNL_iEEG_volt_time_NewSur.append(
        np.mean(iEEG_volt_Time_NewSur[i]["RNL"])
        if i in iEEG_volt_Time_NewSur
        else np.nan
    )
    avShadRNL_iEEG_volt_timeTrue_NewSur.append(
        np.mean(iEEG_volt_TimeTrue_NewSur[i]["RNL"])
        if i in iEEG_volt_TimeTrue_NewSur
        else np.nan
    )

In [ ]:
# tab20
avTrueRNL_iEEG_volt_samp = np.mean([ieeg_volt_Samp[i]["RNL"] for i in x], axis=1)
avShadRNL_iEEG_volt_samp = np.mean([ieeg_volt_Samp[i]["RNLshadow"] for i in x], axis=1)
avTrueRNL_iEEG_volt_time = np.mean([ieeg_volt_Time[i]["RNL"] for i in x], axis=1)
avShadRNL_iEEG_volt_time = np.mean([ieeg_volt_Time[i]["RNLshadow"] for i in x], axis=1)
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1, 2, figsize=(9, 5))
plt.sca(ax[0])
# A[sig_diff_reg>0.05]=np.nan
plt.plot(avTrueRNL_iEEG_volt_samp, avShadRNL_iEEG_volt_samp)
plt.scatter(
    avTrueRNL_iEEG_volt_samp,
    avShadRNL_iEEG_volt_samp,
    c=np.arange(0, 18, 2),
    cmap=palette,
    vmax=18,
    vmin=0,
    label="Fixed Samples",
)
plt.plot(avTrueRNL_iEEG_volt_time, avShadRNL_iEEG_volt_time)
plt.scatter(
    avTrueRNL_iEEG_volt_time,
    avShadRNL_iEEG_volt_time,
    c=np.arange(1, 18, 2),
    cmap=palette,
    vmax=18,
    vmin=0,
    label="Fixed Time",
)
plt.plot(avShadRNL_iEEG_volt_timeTrue_NewSur, avShadRNL_iEEG_volt_time_NewSur)
plt.scatter(
    avShadRNL_iEEG_volt_timeTrue_NewSur,
    avShadRNL_iEEG_volt_time_NewSur,
    c=np.arange(1, 18, 2),
    cmap=palette,
    vmax=18,
    vmin=0,
    label="Fixed Time",
    marker="*",
)
cb = plt.colorbar()
cb.ax.set_ylabel("# regions", rotation=90)
cb.ax.get_yaxis().set_ticks(np.arange(1, 18, 2), band_names)
pts = [max(plt.xlim()[0], plt.ylim()[0]), min(plt.xlim()[1], plt.ylim()[1])]
plt.xlim(plt.xlim())
plt.ylim(plt.ylim())
plt.plot(pts, pts, ":k", lw=1)
plt.xlabel("Data RNL")
plt.ylabel("Shadow RNL")
plt.legend(loc=0)

plt.sca(ax[1])
plt.scatter(
    np.arange(0, 9),
    avTrueRNL_iEEG_volt_samp,
    color=plt.cm.tab20.colors[0],
    label="True",
)
plt.scatter(
    np.arange(0, 9),
    avShadRNL_iEEG_volt_samp,
    color=plt.cm.tab20.colors[2],
    label="Shadow",
)
plt.scatter(
    np.arange(0, 9),
    avTrueRNL_iEEG_volt_samp - avShadRNL_iEEG_volt_samp,
    color=plt.cm.tab20.colors[4],
    label="Difference",
    marker="+",
)
plt.scatter(
    np.arange(0, 9),
    avTrueRNL_iEEG_volt_time,
    color=plt.cm.tab20.colors[1],
    label="True",
)
plt.scatter(
    np.arange(0, 9),
    avShadRNL_iEEG_volt_time,
    color=plt.cm.tab20.colors[3],
    label="Shadow",
)
plt.scatter(
    np.arange(0, 9),
    avTrueRNL_iEEG_volt_time - avShadRNL_iEEG_volt_time,
    color=plt.cm.tab20.colors[5],
    label="Difference",
    marker="+",
)
# plt.xscale("log")
plt.xlabel("# regions")
plt.ylabel("RNL")
ha = [
    Line2D([], [], lw=0, color=plt.cm.tab20.colors[i], marker=m)
    for i, m in zip([0, 2, 4], "oo+")
]
ha += [Line2D([], [], lw=0, color=c, marker="o") for c in ["black", "grey"]]
la = ["True", "Shadow", "Difference", "Fixed Samples", "Fixed Time"]
plt.legend(ha, la, loc=0, frameon=False)
plt.xticks(np.arange(0, 9), band_names)
ax[1].yaxis.set_label_position("right")
ax[1].yaxis.tick_right()
plt.suptitle("Relationship between true and shadow $-$ Scalp Voltage")
plt.show()
pearsonr(avTrueRNL_iEEG_volt_samp, avShadRNL_iEEG_volt_samp), pearsonr(
    avTrueRNL_iEEG_volt_time, avShadRNL_iEEG_volt_time
)

In [ ]:
sr = [4, 8, 12, 15, 18, 30, 44]
# plt.plot(sr, avShadRNL_iEEG_volt_samp_NewSur[:7], label="samp_NewSur")
plt.plot(sr, avShadRNL_iEEG_volt_time_NewSur[:7], label="time_NewSur")
plt.plot(sr, avShadRNL_iEEG_volt_samp[:7], label="p_volt_samp")
plt.plot(sr, avShadRNL_iEEG_volt_time[:7], label="p_volt_time")
plt.legend(loc=0)
plt.show()

In [ ]:
palette = ListedColormap(plt.cm.tab20.colors[:-2])

fig, ax = plt.subplots(1, 2, figsize=(13, 7))
for ax_n, pack in enumerate([("Samples", ieeg_volt_Samp), ("Time", ieeg_volt_Time)]):
    kind, data = pack
    plt.sca(ax[ax_n])
    for i in range(1, 10):
        plt.scatter(
            data[i]["RNL"],
            data[i]["RNLshadow"],
            c=np.full(18, i * 2 + (kind == "Time")),
            cmap=palette,
            vmax=18,
            vmin=0,
            s=5,
        )
    ax[ax_n].set_aspect(1)
    pts = [max(plt.xlim()[0], plt.ylim()[0]), min(plt.xlim()[1], plt.ylim()[1])]
    plt.xlim(plt.xlim())
    plt.ylim(plt.ylim())
    plt.plot(pts, pts, ":k", lw=1)
    plt.xlabel("Data RNL")
    plt.ylabel("Shadow RNL")
    plt.title(kind)
plt.show()


palette = plt.cm.magma
palette.set_bad(color="black")  # Bad values (i.e., masked, set to black 0.8
palette.set_under(color="grey")  # Bad values (i.e., masked, set to black 0.8
palette.set_over(color="grey")  # Bad values (i.e., masked, set to black 0.8
for ax_n, pack in enumerate(
    [("Samples", ieeg_volt_Samp, "o"), ("Time", ieeg_volt_Time, "^")]
):
    kind, data, marker = pack
    corr_ieeg = [pearsonr(data[i]["RNL"], data[i]["RNLshadow"]) for i in range(1, 10)]
    plt.scatter(
        np.arange(0, 9),
        [q[0] for q in corr_ieeg],
        c=[q[1] for q in corr_ieeg],
        cmap=palette,
        vmax=0.05,
        vmin=0,
        marker=marker,
        label=kind,
    )
plt.xticks(np.arange(0, 9), band_names)
plt.legend(loc=0)
plt.show()

In [ ]:
import json, sys
sys.path.append(os.path.abspath(".."))
import mienc.support as ms
from mienc import Corrector, NonLinearEstimator
from matplotlib.colors import LogNorm
with open("/home/raffaelli/NonLinearity/NonLinearityData/iEEG/spikes_by_suject_electrode_4_30.json") as fp:
    sse = {int(k):v for k,v in json.load(fp).items()}
spikes_per_electrode = {s:np.array([len(v[0]) for v in sse[s].values()]) for s in sse}

In [ ]:
def data_from(band, nbins, duration):
    MI_thres = {10: 0.015696, 13: 0.009054, 15: 0.008483, 20: 0.003671, 23: 0.002931}
    corrct = Corrector(
        nbins,
        duration,
        folder_name=f"/home/raffaelli/NonLinearity/NonLinearityResultsNew/iEEG_12276_band{band}_MF_bin{nbins}",
        config="../config.ini",
        cache_dir="../cache/",
        display=False
    )
    corrct.compute_correction()
    ns = []
    rnl = []
    aga = []
    for subj in range(18):
        MI = corrct.correct(
            np.load(
                f"/home/raffaelli/NonLinearity/NonLinearityResultsNew/iEEG_12276_band{band}_MF_bin{nbins}/subject{subj:02}_{nbins}.npy"
            )
        )
        tr = MI[:, 0]
        ga = np.mean(MI[:, 1:], axis=1)
        rnl.append(1 - ga / tr)
        aga.append(ga)
        fe, se = np.triu_indices(24, 1)
        ns.append(
            spikes_per_electrode[subj + 1][fe] + spikes_per_electrode[subj + 1][se]
        )
    ns = np.concatenate(ns)
    aga = np.concatenate(aga)
    rnl = np.concatenate(rnl)
    good = (rnl >= 0) & (rnl <= 1) & (aga > MI_thres[nbins])#0)#
    return ns, rnl, good

In [ ]:
thresh = 0.01
print(np.sqrt(1 - np.exp(-2 * thresh)))
data = [data_from(band, 23, 12276) for band, (duration, nbins) in enumerate(#bins, duration
    [(1116, 10), (2232, 13), (3348, 15), (8432, 20), (12276, 23)], start=1
)]
plt.close()
fig, ax = plt.subplots(2,3,figsize=(15,10), sharex='all', sharey='all')
plt.subplots_adjust(hspace=0, wspace=0)
for band, (ns, rnl, good) in enumerate(data, start=1
):
    his=ax[(band-1)//3, (band-1)%3].hist2d(ns[good], rnl[good], norm=LogNorm())
    r = pearsonr(ns[good], rnl[good])
    ax[(band-1)//3, (band-1)%3].set_title(
        f"{band_names2[band - 1]} - r:{r.statistic:.3} {'*' if r.pvalue<0.01 else ''}", y=1.0, pad=-12
    )
    if (band-1)//3:
        ax[(band-1)//3, (band-1)%3].set_xlabel("# spikes")
    if not (band-1)%3:
        ax[(band-1)//3, (band-1)%3].set_ylabel("RNL")
plt.colorbar(his[-1],ax=ax[1,2]).set_label("# pairs")
ax[1,2].axis("off")
plt.ylim(bottom=0, top=1)
print(ax)
plt.show()

In [ ]:
mat = loadmat("/home/raffaelli/NonLinearity/NonLinearityData/iEEG/ID12_176h.mat")["EEG"]

In [ ]:
3686400/3600, .4836*1024*3600, .5347*1024*3600, 1782743+70000, (.5347-.4836)*3600, 1971118-1782743

In [ ]:
plt.plot(np.arange(150000)/1024, mat[:5,1782743-30000:1782743+120000].T)
plt.ylim(plt.ylim())
plt.vlines(32400/1024, *plt.ylim())

In [ ]:
np.triu_indices(24)

## spikes

# Localisation of non-linearity

In [ ]:
# %matplotlib widget
import json
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import io, h5py, zipfile
import numpy as np
from tqdm import tqdm
import os, sys

sys.path.append(os.path.abspath(".."))
import mienc.support as ms
from mienc import Corrector, NonLinearEstimator
import pandas as pd
from scipy.io import loadmat, savemat
from scipy.stats import ks_1samp, uniform, pearsonr, spearmanr
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
from nilearn import datasets, plotting, image
import nibabel as nib
import pandas as pd
from scipy.spatial import Voronoi, voronoi_plot_2d

aal_atlas_centers = pd.read_csv("../auxiliary_data/AAL_90regions.csv")
aal_atlas_centers_labels = (
    aal_atlas_centers["Labels"].apply(lambda x: x.split()[1]).tolist()
)
atlas_data = datasets.fetch_atlas_aal()
atlas_filename = atlas_data.maps
region_indices = {l: i for l, i in zip(atlas_data["labels"], atlas_data["indices"])}

base_palette = plt.cm.magma
base_palette.set_under(base_palette.get_bad())
base_palette.set_over(base_palette.get_bad())
base_palette.set_bad(color="grey")
base_palette.set_under(color="grey")
cmaplist = [
    base_palette(i) for i in np.arange(11) * 25
]  # np.concatenate([np.arange(6)*25, 151+np.arange(5)*26])
cool_palette = LinearSegmentedColormap.from_list("Custom cmap", cmaplist, 6)
cool_palette.set_under(cool_palette.get_bad())

reg_loc = []
image_data = image.get_data(atlas_filename)
affine = image.load_img(atlas_filename).affine
for l in region_indices:
    if l in aal_atlas_centers_labels:
        reg_loc.append(np.where(image_data == int(region_indices[l])))


def plot_brain(
    in_array,
    title=None,
    cut_coords=None,
    axes=None,
    cmap=base_palette,
    vmin=None,
    vmax=None,
):
    MAXINT = np.iinfo(np.int32).max
    if vmin is None:
        vmin = np.nanmin(in_array)
    if vmax is None:
        vmax = np.nanmax(in_array)
    FACTOR = MAXINT / (max(np.abs(vmin), np.abs(vmax)) + 1)
    values = np.full_like(image_data, np.nan, dtype=np.int32)
    for i in range(90):
        values[reg_loc[i][0], reg_loc[i][1], reg_loc[i][2]] = in_array[i] * FACTOR

    img = nib.nifti1.Nifti1Image(values, affine)
    print(
        FACTOR,
        vmin * FACTOR,
        vmax * FACTOR,
        np.nanmin(in_array[in_array > 0]),
        np.nanmax(in_array),
        np.nanmin(values[values > 0]),
        np.nanmax(values),
    )
    plotting.plot_roi(
        img,
        title=title,
        cut_coords=cut_coords,
        cmap=cmap,
        vmin=vmin * FACTOR,
        vmax=vmax * FACTOR,
        axes=axes,
    )


def plot_cap(
    in_array,
    title=None,
    region_positions=None,
    axes=None,
    cmap=base_palette,
    vmin=None,
    vmax=None,
    plane_distance=10,
):
    source_z = region_positions.z.min()
    diameter = region_positions.z.max() - source_z
    region_positions["XP"] = (
        region_positions.x
        / (region_positions.z - source_z + plane_distance)
        * (diameter + plane_distance)
    )
    region_positions["YP"] = (
        region_positions.y
        / (region_positions.z - source_z + plane_distance)
        * (diameter + plane_distance)
    )
    if vmin is None:
        vmin = np.nanmin(in_array)
    if vmax is None:
        vmax = np.nanmax(in_array)
    vor = Voronoi(
        np.concatenate(
            [
                region_positions[["XP", "YP"]],
                np.transpose(
                    [
                        22 * np.sin(np.linspace(0, 2 * np.pi, 100)),
                        22 * np.cos(np.linspace(0, 2 * np.pi, 100)) - 0.5,
                    ]
                ),
            ]
        )
    )
    normalised = (in_array - vmin) / (vmax - vmin)
    if axes is not None:
        plt.sca(axes)
    plt.gca().set_aspect("equal")

    for i, v in enumerate(normalised):
        region = vor.regions[vor.point_region[i]]
        vertices = np.array([vor.vertices[n] for n in region])
        plt.fill(vertices[:, 0], vertices[:, 1], color=cmap(v))
    if title is not None:
        plt.title(title)
    # voronoi_plot_2d(vor, show_vertices=False, point_size=4)
    # plt.plot(37*np.sin(np.linspace(0,2*np.pi,50)),37*np.cos(np.linspace(0,2*np.pi,50))-5, "-r")


def HolmThresholdFromP(p_values: np.ndarray):
    """Returns the Holm-Bonferroni threshold given an array of p-values.
    NOTA BENE: reject the null hypotheses when **strictly smaller** than this thresold. This corresponds to the *p-value* of the first non-rejected null hypothesis.

    Parameters
    ----------
    p_values : np.ndarray
        The *p-values* to consider, can be N-dimensional, will be flattened.

    Returns
    -------
    float
        The *p-value* of the first non-rejected hypothesis.
    """
    sorted_p = np.sort(p_values.flatten())
    good = sorted_p < 0.05 / (sorted_p.size - np.arange(sorted_p.size))
    which = np.argmin(good)

    if which == 0 and sorted_p[-1] < 0.05:
        return 0.05

    return sorted_p[which]

In [ ]:
MAX_CM = 0
MAX_CM2 = 0
TS_suffix = {"TRUE": "", "SHADOW": "_sha"}


def summarise_localised_non_linearity(
    results_file,
    subset_description,
    subset_identifiers,
    subset_names,
    output_prefix,
    pair_num,
    subj_num,
    elec_num,
    region_positions,
    nbins,
    nsteps,
    cut_position=None,
):
    global MAX_CM, MAX_CM2
    corrct = Corrector(
        nbins,
        nsteps,
        folder_name=os.path.dirname(results_file).format(subset_identifiers[0]),
        config="../config.ini",
        cache_dir="../cache/",
    )
    corrct.compute_correction()
    for subset_id, subset_na in zip(subset_identifiers, subset_names):
        mappa = {}
        siginreg = {}
        for TS in TS_suffix:
            region_quantiles = np.zeros((subj_num, pair_num))
            if os.path.isfile(
                f"../../NonLinearityResultsNew/localised/{output_prefix}{TS_suffix[TS]}_region_quantiles_{subset_id}.npy"
            ):
                region_quantiles = np.load(
                    f"../../NonLinearityResultsNew/localised/{output_prefix}{TS_suffix[TS]}_region_quantiles_{subset_id}.npy"
                )
                print(
                    f"Region quantiles for {TS} {subset_description} {subset_id} loaded"
                )
            else:
                for s in tqdm(range(subj_num), "Subject"):
                    pat = np.load(
                        results_file.format(subset_id, s) + TS_suffix[TS] + ".npy"
                    )
                    true = pat[:, 0]
                    surr = np.sort(pat[:, 1:], 1)
                    for i in range(pair_num):
                        region_quantiles[s, i] = np.searchsorted(
                            surr[i, :], true[i], "left"
                        )
                    np.save(
                        f"../../NonLinearityResultsNew/localised/{output_prefix}{TS_suffix[TS]}_region_quantiles_{subset_id}.npy",
                        region_quantiles,
                    )

            if os.path.isfile(
                f"../../NonLinearityResultsNew/localised/{output_prefix}{TS_suffix[TS]}_ks_stat_{subset_id}.npy"
            ) and os.path.isfile(
                f"../../NonLinearityResultsNew/localised/{output_prefix}{TS_suffix[TS]}_ks_p_{subset_id}.npy"
            ):
                ks_stat = np.load(
                    f"../../NonLinearityResultsNew/localised/{output_prefix}{TS_suffix[TS]}_ks_stat_{subset_id}.npy"
                )
                ks_p = np.load(
                    f"../../NonLinearityResultsNew/localised/{output_prefix}{TS_suffix[TS]}_ks_p_{subset_id}.npy"
                )
                print(f"K-S stats for {TS} {subset_description} {subset_id} loaded")
            else:
                ks_stat = np.zeros(pair_num)
                ks_p = np.zeros(pair_num)
                exp = np.arange(0, 101) / 100
                for i in tqdm(range(pair_num), "Pair"):
                    stat, pval = ks_1samp(
                        region_quantiles[:, i],
                        uniform.cdf,
                        (0, 100),
                        alternative="less",
                    )
                    ks_stat[i] = stat
                    ks_p[i] = pval
                np.save(
                    f"../../NonLinearityResultsNew/localised/{output_prefix}{TS_suffix[TS]}_ks_stat_{subset_id}.npy",
                    ks_stat,
                )
                np.save(
                    f"../../NonLinearityResultsNew/localised/{output_prefix}{TS_suffix[TS]}_ks_p_{subset_id}.npy",
                    ks_p,
                )

            thresh = HolmThresholdFromP(ks_p)
            print("HB thresh:", thresh)

            corrected = np.full(pair_num, np.nan)
            corrected[ks_p < thresh] = ks_stat[ks_p < thresh]
            mappa[TS] = np.zeros([elec_num, elec_num])
            mappa[TS][np.triu_indices(elec_num, 1)] = corrected
            mappa[TS] += mappa[TS].T
            siginreg[TS] = np.sum(mappa[TS] > 0, 1)
            print("Non linear connections:", np.sum(siginreg[TS]) / 2)

        fig, ax = plt.subplots(
            1, 3, gridspec_kw={"width_ratios": [8, 8, 1]}, figsize=(12, 6)
        )
        vmax = max(np.nanmax(mappa["TRUE"]), np.nanmax(mappa["SHADOW"]))
        vmin = min(np.nanmin(mappa["TRUE"]), np.nanmin(mappa["SHADOW"]))
        sc = plt.scatter(
            [np.nan], [np.nan], c=0, vmin=vmin, vmax=vmax, cmap=plt.cm.viridis
        )
        cbar = plt.colorbar(sc, cax=ax[2], shrink=0.3, extend="min")
        cbar.set_label("KS-statistic")

        for i, TS in enumerate(TS_suffix):
            plt.sca(ax[i])
            plt.imshow(mappa[TS], interpolation="none", vmin=vmin, vmax=vmax)
            plt.xlabel(f"{sum(siginreg[TS]>0)}/{elec_num} electrodes")
            plt.title(
                f"{TS} - {int(np.sum(siginreg[TS])/2)} ({100*np.sum(siginreg[TS])/pair_num/2:.3} %) significant pairs"
            )
        plt.suptitle(f"{subset_description} {subset_na}")
        plt.show()

        if region_positions is not None:
            try:
                vmax = max(np.max(siginreg["TRUE"]), np.max(siginreg["SHADOW"]))
                vmin = 1  # min(np.min(siginreg["TRUE"]), np.min(siginreg["SHADOW"]))
            except:
                vmax = 10
                vmin = 1
            print("Siginreg max:", vmax)

            fig, ax = plt.subplots(
                1, 3, gridspec_kw={"width_ratios": [8, 8, 1]}, figsize=(15, 6)
            )
            vmax = max(vmax, 5)
            sc = plt.scatter(
                [np.nan], [np.nan], c=0, cmap=base_palette, vmin=vmin, vmax=vmax
            )
            cbar = plt.colorbar(sc, cax=ax[2], shrink=0.4, extend="min")
            cbar.set_label("# significantly non linear relationships")
            for i, TS in enumerate(TS_suffix):
                plt.sca(ax[i])
                if "aal" in results_file:
                    plot_brain(
                        siginreg[TS],
                        f"AAL 90 regions {TS}",
                        cut_position,
                        ax[i],
                        base_palette,
                        vmin,
                        vmax,
                    )  # (-15,-75,27)
                else:
                    plot_cap(
                        siginreg[TS],
                        f"{TS} {subset_description} {subset_na} - Non-Linearity position",
                        region_positions,
                        ax[i],
                        base_palette,
                        vmin,
                        vmax,
                    )
            plt.suptitle(f"{subset_description} {subset_na}")
            plt.savefig(
                f"{output_prefix}{TS_suffix[TS]}_brain.pdf", bbox_inches="tight"
            )
            plt.show()

## fMRI

In [ ]:
pair_num = 4005
subj_num = 245
elec_num = 90
cut_pos = (-15, -75, 27)
subset_identifiers = ["raw", "mod", "strin"]  # ["strin"]#
subset_names = ["raw", "mod", "strin"]  # ["strin"]#
region_positions = pd.read_csv("../auxiliary_data/AAL_90regions.csv").loc[
    slice(None), ["Labels", "x", "y", "z"]
]
region_positions["Labels"] = region_positions["Labels"].apply(lambda x: x.split()[1])
nbins = 9
nsteps = loadmat(f"/mnt/DATA/Motion_fMRI/Datasets/eso245/eso245_aal_strin_90.mat")[
    "subj_tcs"
].shape[0]

subset_description = "fMRI AAL90 "
output_prefix = "fMRI_AAL90"
results_file = "../../NonLinearityResults/eso245_aal_{}_90_bin9/subject{:02}_9"
ks_stat_true_p = summarise_localised_non_linearity(
    results_file,
    subset_description,
    subset_identifiers,
    subset_names,
    output_prefix,
    pair_num,
    subj_num,
    elec_num,
    region_positions,
    nbins,
    nsteps,
    cut_pos,
)

In [ ]:
pair_num = 4005
subj_num = 245
elec_num = 90
cut_pos = (-15, -75, 27)
subset_identifiers = ["FILT2"]  # ["strin"]#
subset_names = ["FILT2"]  # ["strin"]#
region_positions = pd.read_csv("../auxiliary_data/AAL_90regions.csv").loc[
    slice(None), ["Labels", "x", "y", "z"]
]
region_positions["Labels"] = region_positions["Labels"].apply(lambda x: x.split()[1])
nbins = 9
nsteps = loadmat(f"/mnt/DATA/Motion_fMRI/Datasets/eso245/eso245_aal_strin_90.mat")[
    "subj_tcs"
].shape[0]

subset_description = "fMRI AAL90 "
output_prefix = "fMRI_AAL90"
results_file = "/home/raffaelli/NonLinearity/NonLinearityResultsNew/eso245_aal_raw_90_filtered_{}_bin9/subject{:02}_9"
ks_stat_true_p = summarise_localised_non_linearity(
    results_file,
    subset_description,
    subset_identifiers,
    subset_names,
    output_prefix,
    pair_num,
    subj_num,
    elec_num,
    region_positions,
    nbins,
    nsteps,
    cut_pos,
)

## EEG

In [ ]:
file = zipfile.PyZipFile("../../NonLinearityData/EEG_el_so_BLP_20230714.zip")
archive = h5py.File(io.BytesIO(file.read("info.mat")))
zip_file = zipfile.PyZipFile("../../NonLinearityData/EEG_el_so_BLP_20230714.zip")
infoMat = h5py.File(io.BytesIO(zip_file.read("info.mat")))
vec = np.where(infoMat["elec_in_mask"])[1]
region_positions = (
    pd.read_csv("../auxiliary_data/electrode_positions.csv")
    .loc[vec, ["Labels", "x", "y", "z"]]
    .reset_index()
)

pair_num = 18915
subj_num = 50
elec_num = 195
results_file = (
    "../../NonLinearityResultsNew/NEW_EEG_fixedSamples_band{}_bin10/subject{:02}_10.npy"
)
subset_description = "EEG band "
subset_identifiers = range(1, 10)
subset_names = [
    r"$\delta$",
    r"$\theta$",
    r"$\alpha$",
    r"$\beta_{LOW}$",
    r"$\beta_{MID}$",
    r"$\beta_{HIGH}$",
    r"$\gamma$",
    r"$\beta_{ALL}$",
    r"$1-40 Hz$",
]
output_prefix = "electrodeEEG"
nbins = 10
nsteps = loadmat(
    f"/home/raffaelli/NonLinearity/NonLinearityData/EEG_el_so_BLP_NEW/NEW_EEG_fixedSamples_band1.mat"
)["EEG"].shape[0]
summarise_localised_non_linearity(
    results_file,
    subset_description,
    subset_identifiers,
    subset_names,
    output_prefix,
    pair_num,
    subj_num,
    elec_num,
    region_positions,
    nbins,
    nsteps,
)

# Sources of non-linearity

## fMRI

## EEG

## iEEG

## spikes